# 05 — Feature Engineering

**Input:** bảng `public.cleaned_city_temperature` được bàn giao từ `03_data_cleaning.ipynb`.

**Output chính:** bảng `public.city_temperature_features` trong PostgreSQL. Các file `data/processed/feature_engineered_data.csv.gz`, `data/processed/feature_metadata.json`, `data/processed/feature_statistics.csv.gz` và `data/sample/feature_sample.csv` là bản bàn giao bổ sung cho kiểm tra và demo.

Notebook giữ nguyên grain **một thành phố tại một tháng**. Mỗi dòng được định danh bằng khóa nghiệp vụ `observation_date + city_name + country_name + latitude + longitude`; biến mục tiêu là `city_average_temperature`.

Kịch bản được chọn là **dự báo một tháng kế tiếp**: tại thời điểm dự báo, mô hình được phép dùng thuộc tính tĩnh của địa điểm và dữ liệu các tháng trước, nhưng không được dùng phép đo của chính tháng đang dự báo.

## 1. Mục tiêu, vai trò và pipeline Feature Engineering

### 1.1. Mục tiêu của Notebook

Notebook 05 biến tập dữ liệu đã làm sạch thành một tập **đặc trưng (feature)** có căn cứ, không rò rỉ dữ liệu và sẵn sàng để Notebook 06 huấn luyện mô hình. Các công việc chính gồm:

1. Chuyển các quan sát định lượng của Notebook 04 thành ý tưởng feature cụ thể.
2. Xây dựng tám nhóm feature: thời gian, địa lý, tương tác, khí hậu theo vị trí, trễ/trượt, bối cảnh toàn cầu, chất lượng dữ liệu và mã hóa quốc gia.
3. Fit các thống kê cần thiết chỉ trên feature-fit, dùng leave-one-out và giữ nguyên trật tự thời gian.
4. Kiểm tra kiểu dữ liệu, giá trị thiếu, tính hợp lệ của one-hot, đa cộng tuyến và nguy cơ leakage.
5. Loại trước các cột không hợp lệ về mặt ngữ nghĩa hoặc không khả dụng tại thời điểm dự báo.
6. Bàn giao toàn bộ feature ứng viên hợp lệ cùng metadata, split thời gian và bảng thống kê cho Notebook 06.

Notebook này **không huấn luyện mô hình, không tính model-based feature importance và không dùng kết quả mô hình để chọn feature**. Các công việc đó thuộc Notebook 06 với Linear Regression, Random Forest và XGBoost.

### 1.2. Vai trò của Feature Engineering trong dự án

Notebook 04 cho thấy các biến thô trong bảng sạch **không phản ánh đúng bản chất khí hậu** của dữ liệu:

- `month` và `quarter` chỉ có tương quan tuyến tính `r ≈ 0.10` với nhiệt độ, dù biên độ mùa vụ lên tới **13.04 °C**. Nguyên nhân là quan hệ tháng–nhiệt độ có tính **tuần hoàn**: tháng 12 và tháng 1 liền kề về khí hậu nhưng cách xa nhau về mặt số học.
- `year` và `decade` chỉ có `r ≈ 0.048`, dù xu hướng ấm lên là thật ở mức **xấp xỉ +0,027 °C/năm**. Tín hiệu dài hạn này bị biên độ mùa vụ và biên độ vĩ độ (lớn hơn nhiều) che lấp.
- `latitude` là driver địa lý mạnh nhất (`r = -0.467`) nhưng vì có dấu, nó trộn lẫn hai thông tin khác nhau: khoảng cách tới xích đạo và bán cầu.

Feature Engineering là bước biểu diễn lại các thông tin đó dưới dạng mà mô hình học được: mã hóa tuần hoàn cho mùa vụ, biến trôi thời gian cho xu hướng, khoảng cách tuyệt đối tới xích đạo cho vĩ độ. Đây cũng là bước duy nhất trong pipeline chịu trách nhiệm **chặn rò rỉ dữ liệu** trước khi mô hình được huấn luyện.

### 1.3. Kiến trúc pipeline

```text
public.cleaned_city_temperature  (5.579.085 dòng × 32 cột)
   ↓ chỉ đọc 14 cột cần dùng từ PostgreSQL
Bảng làm việc theo grain City–month
   ↓ chia thời gian: feature-fit → validation → test
8 nhóm feature an toàn theo kịch bản dự báo một tháng kế tiếp
   ↓
Kiểm tra schema + missing + one-hot + multicollinearity + leakage
   ↓
Tập feature ứng viên hợp lệ + metadata + split thời gian
   ↓
public.city_temperature_features
   ↓
Notebook 06 — train Linear Regression / Random Forest / XGBoost,
đánh giá mô hình, feature importance và lựa chọn feature
```

Các nguyên tắc thiết kế chính:

- **PostgreSQL là nguồn duy nhất.** Notebook dừng nếu thiếu `.env`, không kết nối được hoặc chưa có bảng sạch.
- **Không sử dụng test để ra quyết định.** Notebook 05 chỉ gắn nhãn split; Notebook 06 chịu trách nhiệm fit trên train, chọn mô hình bằng validation và chỉ báo cáo cuối trên test.
- **Target statistics chống self-target leakage.** Các dòng feature-fit dùng leave-one-out; validation/test chỉ dùng thống kê học từ giai đoạn kết thúc năm 1983.
- **Feature động chỉ nhìn quá khứ.** Lag nhiệt độ, chỉ số toàn cầu và uncertainty đều dùng tháng trước.
- **Feature suy ra trực tiếp từ target không được xuất.** `temp_anomaly_vs_climatology` chỉ phục vụ insight ở Mục 8.
- **Modeling được tách khỏi Feature Engineering.** Notebook 05 không gọi `.fit()` cho bất kỳ mô hình nào.

## 2. Chuẩn bị môi trường và cấu hình

### 2.1. Nạp các thư viện cần thiết

`pandas` và `numpy` phục vụ xây dựng và kiểm tra feature; `matplotlib` và `seaborn` dùng cho các insight mô tả; `SQLAlchemy` kết hợp `psycopg2` đọc/ghi PostgreSQL; `python-dotenv` nạp cấu hình từ `.env` mà không ghi mật khẩu vào notebook.

Notebook 05 không cần `scikit-learn`, `xgboost` hoặc `shap` vì không huấn luyện và không giải thích mô hình. Các thư viện modeling sẽ được cài đặt và sử dụng ở Notebook 06.

In [42]:
import gc
import json
import os
import re
import shutil
import unicodedata
import warnings
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 170)
pd.set_option('display.float_format', lambda value: f'{value:,.4f}')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (13, 5.5)
plt.rcParams['font.size'] = 10
warnings.filterwarnings('ignore', category=FutureWarning)

print(f'pandas {pd.__version__} · numpy {np.__version__}')

pandas 3.0.3 · numpy 2.5.0


Các thư viện trong cell trên chỉ phục vụ Feature Engineering, kiểm tra chất lượng và trực quan mô tả. Việc không import thư viện mô hình là một kiểm soát phạm vi: nếu trong Notebook 05 xuất hiện `LinearRegression`, `RandomForestRegressor`, `XGBRegressor`, `.fit()` hoặc SHAP thì phần đó phải được chuyển sang Notebook 06.

### 2.2. Xác định thư mục dự án và đầu ra

Notebook có thể được mở từ thư mục gốc, `notebooks_v1/` hoặc `notebooks_v2/`. Hàm dưới tìm project root dựa trên hai thư mục ổn định `data/` và `SQL/`, sau đó chuẩn bị các thư mục đầu ra.

In [43]:
def find_project_root(start: Path) -> Path:
    """Tìm thư mục gốc chứa cả data/ và SQL/."""
    for candidate in (start, *start.parents):
        if (candidate / 'data').is_dir() and (candidate / 'SQL').is_dir():
            return candidate
    raise FileNotFoundError('Không tìm thấy project root chứa data/ và SQL/.')


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
SAMPLE_DIR = PROJECT_ROOT / 'data' / 'sample'
REPORT_IMAGES_DIR = PROJECT_ROOT / 'reports' / 'images'
for directory in (PROCESSED_DIR, SAMPLE_DIR, REPORT_IMAGES_DIR):
    directory.mkdir(parents=True, exist_ok=True)

CLEANED_TABLE = 'cleaned_city_temperature'
FEATURE_TABLE = 'city_temperature_features'
METADATA_PATH = PROCESSED_DIR / 'feature_metadata.json'
FEATURE_STATS_PATH = PROCESSED_DIR / 'feature_statistics.csv.gz'
SAMPLE_PATH = SAMPLE_DIR / 'feature_sample.csv'

print('Project root   :', PROJECT_ROOT)
print('Input table    :', f'public.{CLEANED_TABLE}')
print('Processed dir  :', PROCESSED_DIR)
print('Report images  :', REPORT_IMAGES_DIR)

Project root   : E:\FPT\HocKy3\PROJECT_1\PROJECT\Global-Surface-Temperature-Analysis
Input table    : public.cleaned_city_temperature
Processed dir  : E:\FPT\HocKy3\PROJECT_1\PROJECT\Global-Surface-Temperature-Analysis\data\processed
Report images  : E:\FPT\HocKy3\PROJECT_1\PROJECT\Global-Surface-Temperature-Analysis\reports\images


Đường dẫn in ra phải trỏ tới đúng project hiện tại. Notebook không đọc, ghi hoặc thay đổi bất kỳ file nào trong `data/raw/`.

### 2.3. Tham số cấu hình của notebook

Toàn bộ tham số được gom vào một cell để dễ kiểm soát:

- `FAST_MODE`: giới hạn số dòng truy vấn sau khi đã sắp xếp vị trí–thời gian, dùng để kiểm tra nhanh logic feature.
- `FEATURE_FIT_CUTOFF_YEAR = 1983`: giai đoạn duy nhất được dùng để học target statistics.
- `VALIDATION_START_YEAR = 1984` và `TRAIN_CUTOFF_YEAR = 1993`: xác định nhãn split để Notebook 06 sử dụng; Notebook 05 không đánh giá mô hình trên các split này.
- `FEATURE_AUDIT_SAMPLE_ROWS`: kích thước mẫu feature-fit dùng cho kiểm tra đa cộng tuyến, không dùng để train model.
- `MULTICOLLINEARITY_THRESHOLD`: ngưỡng cảnh báo cặp feature có tương quan tuyệt đối rất cao; đây chỉ là chẩn đoán, không tự động loại feature.
- `OUTPUT_COMPRESSION`: điều khiển bản CSV kiểm tra; bảng PostgreSQL vẫn là đầu ra chính.

In [ ]:
# --- Chế độ chạy ---------------------------------------------------------
FAST_MODE = False          # True: chạy thử nhanh trên một phần dữ liệu
FAST_MODE_ROWS = 400_000   # số dòng đầu theo thứ tự vị trí–thời gian

# --- Chia dữ liệu theo thời gian -----------------------------------------
FEATURE_FIT_CUTOFF_YEAR = 1983  # chỉ giai đoạn này học target statistics
VALIDATION_START_YEAR = 1984    # nhãn validation để Notebook 06 sử dụng
TRAIN_CUTOFF_YEAR = 1993        # test bắt đầu từ 1994

# --- Kiểm tra chất lượng feature -----------------------------------------
FEATURE_AUDIT_SAMPLE_ROWS = 200_000
MULTICOLLINEARITY_THRESHOLD = 0.95
RANDOM_SEED = 42

# --- Đầu ra --------------------------------------------------------------
OUTPUT_COMPRESSION = 'gzip'
SAMPLE_ROWS = 50_000
TARGET_COLUMN = 'city_average_temperature'

rng = np.random.default_rng(RANDOM_SEED)
OUTPUT_PATH = PROCESSED_DIR / (
    'feature_engineered_data.csv.gz' if OUTPUT_COMPRESSION == 'gzip'
    else 'feature_engineered_data.csv'
)

if not FEATURE_FIT_CUTOFF_YEAR < VALIDATION_START_YEAR <= TRAIN_CUTOFF_YEAR:
    raise ValueError('Các mốc fit/validation/train không đúng thứ tự thời gian.')

print(f'FAST_MODE                     : {FAST_MODE}')
print(f'FEATURE_FIT_CUTOFF_YEAR       : {FEATURE_FIT_CUTOFF_YEAR}')
print(f'VALIDATION_START_YEAR         : {VALIDATION_START_YEAR}')
print(f'TRAIN_CUTOFF_YEAR             : {TRAIN_CUTOFF_YEAR}')
print(f'FEATURE_AUDIT_SAMPLE_ROWS     : {FEATURE_AUDIT_SAMPLE_ROWS:,}')
print(f'MULTICOLLINEARITY_THRESHOLD   : {MULTICOLLINEARITY_THRESHOLD}')
print(f'Target                        : {TARGET_COLUMN}')
print(f'Output                        : {OUTPUT_PATH.name}')

Nếu `FAST_MODE = True`, mọi số liệu in ra sau đó chỉ mang tính kiểm tra logic và **không được dùng cho báo cáo**. Kết quả chính thức phải chạy với `FAST_MODE = False`.

### 2.4. Kết nối PostgreSQL bắt buộc

Notebook 05 chỉ đọc dữ liệu từ `public.cleaned_city_temperature` trong PostgreSQL để tuân thủ pipeline đã thống nhất từ Notebook 02–04. Không có nhánh fallback CSV: nếu database không khả dụng, notebook dừng và báo rõ lỗi để tránh vô tình dùng một bản dữ liệu khác.

Tạo file `.env` ở project root từ `.env.example`:

```text
DB_HOST=localhost
DB_PORT=5432
DB_NAME=climate_db
DB_USER=postgres
DB_PASSWORD=<mật khẩu PostgreSQL trên máy đang chạy>
```

In [ ]:
ENV_PATH = PROJECT_ROOT / '.env'
if not ENV_PATH.is_file():
    raise FileNotFoundError(
        f'Không tìm thấy {ENV_PATH}. Notebook 05 chỉ đọc dữ liệu từ PostgreSQL.'
    )

from dotenv import load_dotenv
from sqlalchemy import URL, create_engine, text

load_dotenv(ENV_PATH, override=False)
required_env = ('DB_HOST', 'DB_PORT', 'DB_NAME', 'DB_USER', 'DB_PASSWORD')
missing_env = [name for name in required_env if not os.getenv(name)]
if missing_env:
    raise ValueError(f'.env thiếu các biến: {missing_env}')

DB_ENGINE = create_engine(
    URL.create(
        drivername='postgresql+psycopg2',
        username=os.environ['DB_USER'],
        password=os.environ['DB_PASSWORD'],
        host=os.environ['DB_HOST'],
        port=int(os.environ['DB_PORT']),
        database=os.environ['DB_NAME'],
    ),
    pool_pre_ping=True,
    connect_args={'connect_timeout': 10},
)

check_sql = text(
    'SELECT current_database() AS database_name, '
    f"to_regclass('public.{CLEANED_TABLE}') IS NOT NULL AS source_table_exists"
)
with DB_ENGINE.connect() as connection:
    db_check = pd.read_sql_query(check_sql, connection)

display(db_check)
if not bool(db_check.loc[0, 'source_table_exists']):
    raise RuntimeError(
        f'Không tìm thấy public.{CLEANED_TABLE}. Hãy hoàn thành Notebook 03 trước.'
    )

DB_AVAILABLE = True
DATA_SOURCE = f'POSTGRESQL · public.{CLEANED_TABLE}'
print('PostgreSQL:', f"Đã kết nối {db_check.loc[0, 'database_name']}.public.{CLEANED_TABLE}")

Cell kết nối chỉ thực hiện truy vấn kiểm tra database và sự tồn tại của bảng nguồn; mật khẩu không được hiển thị. Notebook chỉ tiếp tục khi kết nối thành công và `public.cleaned_city_temperature` tồn tại.

## 3. Đọc dữ liệu đã làm sạch

### 3.1. Xác nhận data contract từ Notebook 03

Theo Notebook 03, bảng sạch phải có **5.579.085 dòng**, **32 cột**, **50 quốc gia**, thời gian từ `1863-01-01` đến `2013-09-01` và không còn dòng nào thiếu biến mục tiêu. Các hằng số dưới đây được dùng để đối chiếu sau khi nạp dữ liệu.

In [ ]:
EXPECTED_CLEAN_ROWS = 5_579_085
EXPECTED_COUNTRIES = 50
EXPECTED_MIN_DATE = pd.Timestamp('1863-01-01')
EXPECTED_MAX_DATE = pd.Timestamp('2013-09-01')

BUSINESS_KEY_COLUMNS = [
    'observation_date', 'city_name', 'country_name', 'latitude', 'longitude',
]
LOCATION_KEY_COLUMNS = ['city_name', 'country_name', 'latitude', 'longitude']

print(f'Kỳ vọng {EXPECTED_CLEAN_ROWS:,} dòng · {EXPECTED_COUNTRIES} quốc gia · '
      f'{EXPECTED_MIN_DATE.date()} → {EXPECTED_MAX_DATE.date()}')

### 3.2. Chọn cột cần dùng và kiểu dữ liệu tối ưu

Bảng sạch có 32 cột nhưng Notebook 05 chỉ đọc **14 cột** để giảm thời gian truyền và bộ nhớ.

Các cột nhiệt độ cùng tháng chỉ được đọc khi cần tạo phiên bản lịch sử an toàn:

- `land_average_temperature` là nguồn tạo `land_temperature_lag_1`; bản cùng tháng không được đưa vào feature bàn giao.
- `city_average_temperature_uncertainty` là nguồn tạo `city_uncertainty_lag_1`; uncertainty cùng tháng không được dùng dự báo.

`country_average_temperature`, `major_city_average_temperature`, `land_max_temperature`, `land_min_temperature` và `land_and_ocean_average_temperature` không cần đọc. Chúng là phép đo cùng tháng mục tiêu và đã bị loại theo data contract ở Mục 7.3.

In [ ]:
CALENDAR_COLUMNS = ['year', 'month', 'quarter', 'decade']
TEMPERATURE_COLUMNS = [
    TARGET_COLUMN,                              # biến mục tiêu
    'city_average_temperature_uncertainty',     # nguồn tạo uncertainty lag 1
    'land_average_temperature',                 # nguồn tạo chỉ số toàn cầu lag 1
]
FLAG_COLUMNS = ['is_major_city', 'city_temperature_iqr_outlier']

USED_COLUMNS = BUSINESS_KEY_COLUMNS + CALENDAR_COLUMNS + TEMPERATURE_COLUMNS + FLAG_COLUMNS

# Kiểu dữ liệu nhỏ nhất còn biểu diễn đúng giá trị (xem CHECK constraint của Notebook 03).
READ_DTYPES = {
    'year': 'int16', 'month': 'int8', 'quarter': 'int8', 'decade': 'int16',
    'city_name': 'category', 'country_name': 'category',
    'latitude': 'float32', 'longitude': 'float32',
    **{column: 'float32' for column in TEMPERATURE_COLUMNS},
}

print(f'Đọc {len(USED_COLUMNS)}/32 cột:')
for index, column in enumerate(USED_COLUMNS, start=1):
    print(f'  {index:2d}. {column}')

`city_name` và `country_name` dùng `category` vì chỉ có 3.070 và 50 giá trị duy nhất trên hơn 5,5 triệu dòng — hạ từ chuỗi Python xuống mã số nguyên tiết kiệm phần lớn bộ nhớ của hai cột này. Các cột nhiệt độ dùng `float32` thay vì `float64`: dữ liệu gốc chỉ có 3 chữ số thập phân nên không mất thông tin, mà giảm một nửa bộ nhớ.

### 3.3. Nạp dữ liệu theo từng chunk

Dữ liệu được truy vấn từ PostgreSQL theo các chunk 500.000 dòng để giảm bộ nhớ đỉnh. Truy vấn có `ORDER BY country_name, city_name, latitude, longitude, observation_date`, giúp chuỗi của từng vị trí luôn đúng thứ tự trước khi tạo lag.

In [ ]:
CHUNK_SIZE = 500_000


def compact_chunk(chunk: pd.DataFrame) -> pd.DataFrame:
    """Chuẩn hóa kiểu dữ liệu của một chunk trước khi ghép DataFrame lớn."""
    chunk['observation_date'] = pd.to_datetime(
        chunk['observation_date'], errors='raise'
    )
    for column, dtype in READ_DTYPES.items():
        if column in chunk.columns and chunk[column].dtype.name != dtype:
            chunk[column] = chunk[column].astype(dtype)
    for column in FLAG_COLUMNS:
        if chunk[column].dtype != bool:
            chunk[column] = chunk[column].astype('bool')
    return chunk


limit_clause = f' LIMIT {FAST_MODE_ROWS}' if FAST_MODE else ''
query = text(
    f"SELECT {', '.join(USED_COLUMNS)} FROM public.{CLEANED_TABLE} "
    'ORDER BY country_name, city_name, latitude, longitude, observation_date'
    f'{limit_clause}'
)

parts = []
with DB_ENGINE.connect() as connection:
    for chunk in pd.read_sql_query(query, connection, chunksize=CHUNK_SIZE):
        parts.append(compact_chunk(chunk))

if not parts:
    raise RuntimeError(f'public.{CLEANED_TABLE} không có dữ liệu.')

feature_df = pd.concat(parts, ignore_index=True)[USED_COLUMNS]

# LIMIT có thể cắt dở chuỗi của vị trí cuối. Loại vị trí đó trước mọi phép
# chia thời gian để DataFrame và các mask luôn cùng chỉ mục.
if FAST_MODE:
    last_location = feature_df.iloc[-1][LOCATION_KEY_COLUMNS]
    last_location_mask = pd.Series(True, index=feature_df.index)
    for column in LOCATION_KEY_COLUMNS:
        last_location_mask &= feature_df[column].eq(last_location[column])
    if (~last_location_mask).any():
        feature_df = feature_df.loc[~last_location_mask].reset_index(drop=True)

print(f'Nguồn dữ liệu : {DATA_SOURCE}')
print(f'Đã nạp        : {len(feature_df):,} dòng × {feature_df.shape[1]} cột')

Dòng `Nguồn dữ liệu` phải luôn là `POSTGRESQL · public.cleaned_city_temperature`. Số cột đọc vào phải bằng 16; nếu kết nối hoặc data contract sai, notebook dừng thay vì chuyển nguồn.

### 3.4. Kiểm tra data contract, grain và bộ nhớ

Trước khi xây dựng đặc trưng, notebook kiểm tra ba điều kiện. Hai điều kiện đầu là **lỗi cấu trúc** và sẽ làm dừng notebook, vì mọi đặc trưng trễ phía sau đều dựa vào giả định grain duy nhất:

1. Khóa nghiệp vụ City–month phải duy nhất.
2. Biến mục tiêu không được có giá trị thiếu.

Điều kiện thứ ba — row count, số quốc gia, khoảng thời gian — chỉ **cảnh báo**, vì `FAST_MODE` cố tình đọc ít dòng hơn và người dùng có thể chạy lại pipeline với phạm vi khác.

In [ ]:
duplicate_key_rows = int(feature_df.duplicated(subset=BUSINESS_KEY_COLUMNS).sum())
missing_target_rows = int(feature_df[TARGET_COLUMN].isna().sum())
if duplicate_key_rows:
    raise RuntimeError(
        f'Có {duplicate_key_rows:,} dòng trùng khóa nghiệp vụ City–month. '
        'Đặc trưng trễ yêu cầu grain duy nhất; hãy kiểm tra lại Notebook 03.'
    )
if missing_target_rows:
    raise RuntimeError(
        f'Có {missing_target_rows:,} dòng thiếu {TARGET_COLUMN}. '
        'Notebook 03 phải loại các dòng này trước khi bàn giao.'
    )

actual_countries = int(feature_df['country_name'].nunique())
actual_locations = int(feature_df.groupby(LOCATION_KEY_COLUMNS, observed=True).ngroups)
contract_report = pd.DataFrame(
    {
        'Expected': [EXPECTED_CLEAN_ROWS, EXPECTED_COUNTRIES,
                     EXPECTED_MIN_DATE.date(), EXPECTED_MAX_DATE.date()],
        'Actual': [len(feature_df), actual_countries,
                   feature_df['observation_date'].min().date(),
                   feature_df['observation_date'].max().date()],
    },
    index=['Row count', 'Country count', 'Min observation date', 'Max observation date'],
)
contract_report['Match'] = contract_report['Expected'].astype(str) == contract_report['Actual'].astype(str)
display(contract_report)

print(f'Số vị trí (city+country+toạ độ): {actual_locations:,}')
print(f'Bộ nhớ đang dùng               : '
      f'{feature_df.memory_usage(deep=True).sum() / 1024 ** 2:,.1f} MB')
print(f'Khóa nghiệp vụ duy nhất        : PASS ({duplicate_key_rows} dòng trùng)')
print(f'Biến mục tiêu đầy đủ           : PASS ({missing_target_rows} dòng thiếu)')

if not contract_report['Match'].all():
    if FAST_MODE:
        print('\nLƯU Ý: FAST_MODE đang bật nên row count nhỏ hơn contract là bình thường.')
    else:
        print('\nCẢNH BÁO: dữ liệu không khớp contract của Notebook 03. '
              'Hãy kiểm tra lại phiên bản bảng sạch trước khi dùng kết quả cho báo cáo.')

Hai dòng `PASS` xác nhận grain City–month duy nhất và biến mục tiêu đầy đủ — đây là điều kiện bắt buộc để các đặc trưng trễ ở Mục 5.6 tra cứu đúng theo lịch. Bộ nhớ in ra nên ở mức vài trăm MB nhờ các kiểu dữ liệu đã hạ ở Mục 3.2; nếu con số này lên tới nhiều GB, rất có thể `READ_DTYPES` chưa được áp dụng.

In [ ]:
display(feature_df.head())
display(feature_df.dtypes.to_frame('dtype'))

Bảng mẫu cho thấy dữ liệu đã ở đúng grain City–month với các cột nghiệp vụ trực tiếp. Cột `major_city_average_temperature` phần lớn là `NaN` — đúng như Notebook 04 mô tả: chỉ khoảng 2,7% dòng thuộc nhóm Major City.

## 4. Ý tưởng xây dựng Feature từ kết quả EDA

### 4.1. Chuyển kết quả EDA thành ý tưởng Feature

Mục này chuyển các phát hiện quan trọng của Notebook 04 thành **tập feature ứng viên** cho bài toán dự báo nhiệt độ trung bình của thành phố ở **tháng kế tiếp**. Một biến chỉ được đề xuất khi đáp ứng đồng thời ba điều kiện:

1. Có cơ sở từ EDA hoặc từ đặc điểm vật lý của dữ liệu khí hậu.
2. Có thể biết tại thời điểm dự báo, hoặc chỉ sử dụng lịch sử đã xảy ra.
3. Có thể tính nhất quán cho tập fit, validation và test mà không làm rò rỉ biến mục tiêu.

| Nhóm ý tưởng | Feature ứng viên chính | Nguồn thông tin | Thời điểm có thể sử dụng |
|---|---|---|---|
| Chu kỳ thời gian | `month_sin`, `month_cos`, `climatic_month_sin`, `climatic_month_cos`, `years_since_start` | Ngày dự báo | Có sẵn trước khi dự báo |
| Địa lý | `latitude`, `abs_latitude`, `hemisphere_north`, `longitude`, `is_major_city` | Thuộc tính vị trí | Tĩnh, có sẵn trước khi dự báo |
| Phân loại quốc gia | `country_ohe__*` | `country_name` | Fit danh sách quốc gia trên tập fit; cố định cho validation/test |
| Tương tác khí hậu | `abslat_x_month_sin`, `abslat_x_month_cos`, `abslat_x_years` | Thời gian × địa lý | Có sẵn trước khi dự báo |
| Khí hậu chuẩn theo địa điểm | `loc_month_climatology`, `loc_mean_temperature`, `loc_temperature_std`, `loc_climatology_imputed` | Thống kê chỉ học từ tập fit | Có sau bước fit feature |
| Lịch sử nhiệt độ thành phố | `temp_lag_1`, `temp_lag_12`, `temp_roll_mean_12`, `temp_roll_std_12`, `temp_anomaly_lag_12` | Các tháng trước | Chỉ dùng lịch sử đã quan sát |
| Bối cảnh khí hậu toàn cầu | `land_temperature_lag_1`, `land_anomaly_lag_1` | Nhiệt độ toàn cầu tháng trước | Chỉ dùng lịch sử đã quan sát |
| Chất lượng phép đo | `city_uncertainty_lag_1` | Độ bất định tháng trước | Chỉ dùng lịch sử đã quan sát |

Các feature trên là **ứng viên hợp lệ về mặt kỹ thuật**. Notebook 05 kiểm tra schema, missing, one-hot, trật tự thời gian, đa cộng tuyến và leakage rồi bàn giao toàn bộ cho Notebook 06. Notebook 06 mới huấn luyện Linear Regression, Random Forest, XGBoost, đánh giá feature importance và quyết định feature nào thực sự hữu ích cho mô hình.

#### 4.1.1. Feature biểu diễn chu kỳ thời gian

EDA cho thấy nhiệt độ thay đổi rõ rệt theo tháng. Tuy nhiên, nếu dùng trực tiếp số tháng từ 1 đến 12, mô hình sẽ hiểu tháng 12 cách xa tháng 1, trong khi hai tháng này thực tế nằm cạnh nhau trên chu kỳ năm. Vì vậy, tháng được mã hóa bằng hai thành phần tuần hoàn:


$$\text{month\_sin}=\sin\left(2\pi\frac{month}{12}\right),\qquad
\text{month\_cos}=\cos\left(2\pi\frac{month}{12}\right)$$


Hai feature `month_sin` và `month_cos` giúp mô hình nhận biết vị trí của một tháng trên vòng tuần hoàn và giữ được quan hệ gần nhau giữa tháng 12 và tháng 1.

**Vì sao phải dùng đồng thời cả `sin` và `cos`?** Nếu chỉ dùng một hàm, một số tháng khác nhau có thể có cùng giá trị hoặc gần như cùng giá trị. Chẳng hạn, với mã hóa `sin`, tháng 1 và tháng 5 cùng có giá trị xấp xỉ `0.50`; tháng 7 và tháng 11 cùng có giá trị xấp xỉ `-0.50`; tháng 6 và tháng 12 cùng có giá trị gần `0`. Khi đó, mô hình bị mất thông tin về **pha mùa**: biết một điểm cao/thấp trên đồ thị sin nhưng không biết nó nằm ở phía nào của chu kỳ năm.

Kết hợp hai hàm tương đương với biểu diễn mỗi tháng bằng một tọa độ trên đường tròn. Ví dụ, tháng 1 có cặp xấp xỉ `(0.50, 0.87)`, tháng 4 là `(0.87, -0.50)`, tháng 7 là `(-0.50, -0.87)` và tháng 10 là `(-0.87, 0.50)`. Nhờ hai tọa độ này, mỗi tháng có biểu diễn riêng; đồng thời tháng 12 vẫn nằm gần tháng 1 trong không gian feature. Vì vậy, `sin` + `cos` là cặp mã hóa chu kỳ đầy đủ, còn chỉ dùng một biến sẽ làm mất một phần thông tin mùa vụ.

Mùa tại hai bán cầu lệch nhau khoảng sáu tháng. Vì vậy, `climatic_month_sin` và `climatic_month_cos` được xây dựng từ **tháng khí hậu**: giữ nguyên tháng ở Bắc bán cầu và dịch sáu tháng ở Nam bán cầu. Cách biểu diễn này giúp cùng một pha của feature gần với cùng một mùa khí hậu hơn, thay vì chỉ cùng tháng lịch.

`years_since_start` biểu diễn số năm kể từ mốc đầu dữ liệu. Feature này cho phép mô hình học xu hướng dài hạn dễ hơn so với việc dùng trực tiếp năm có giá trị lớn. Tuy nhiên, nó chỉ mô tả trục thời gian; hệ số tăng theo thời gian không được diễn giải tự động là quan hệ nhân quả.

Toàn bộ nhóm này được xếp vào loại **static/calendar features** vì ngày cần dự báo đã được biết trước, không cần truy cập nhiệt độ của chính tháng cần dự báo.

#### 4.1.2. Feature địa lý

EDA cho thấy vĩ độ có quan hệ đáng kể với nhiệt độ: khu vực càng xa xích đạo thường có nền nhiệt thấp hơn và biên độ mùa lớn hơn. Nhóm địa lý vì thế gồm:

- `latitude`: giữ dấu của vĩ độ để phân biệt Bắc bán cầu và Nam bán cầu.
- `abs_latitude`: khoảng cách tuyệt đối tới xích đạo, phù hợp với cơ chế bức xạ Mặt Trời và nền nhiệt theo đới khí hậu.
- `hemisphere_north`: cờ nhị phân cho biết địa điểm thuộc Bắc bán cầu; hỗ trợ mô hình phân biệt pha mùa giữa hai bán cầu.
- `longitude`: cung cấp vị trí theo hướng đông–tây. Bản thân kinh độ không quyết định trực tiếp nhiệt độ nhưng có thể hỗ trợ phân biệt các vùng có khí hậu khác nhau.
- `is_major_city`: cho biết quan sát có thuộc danh sách thành phố lớn hay không. Đây chỉ là thuộc tính tĩnh của thành phố, **không phải nhiệt độ của bảng major city trong cùng tháng**.

Các cột tọa độ đã được chuẩn hóa từ ký hiệu `N/S/E/W` sang số có dấu ở Notebook 03. Vì chúng là thuộc tính cố định của địa điểm, nhóm này có thể sử dụng an toàn ở mọi thời điểm dự báo.

`country_name` không được đưa trực tiếp vào mô hình dưới dạng chuỗi mà được chuyển thành các cột one-hot ở Mục 5.9. Với 50 quốc gia, cách mã hóa này chỉ tạo 49 cột sau khi bỏ nhóm tham chiếu nên vẫn phù hợp với Linear Regression, Random Forest và XGBoost. `city_name` không được one-hot: hơn 3.000 thành phố sẽ tạo hơn 3.000 cột rất thưa, làm tăng mạnh bộ nhớ, khuyến khích mô hình ghi nhớ từng thành phố và không hỗ trợ tốt địa điểm chưa thấy khi train. Tên thành phố vẫn được giữ để tạo khóa địa điểm, tra cứu lịch sử và hiển thị kết quả; thông tin khí hậu riêng của thành phố được biểu diễn bằng tọa độ, lag và climatology theo vị trí.

#### 4.1.3. Feature tương tác giữa thời gian và địa lý

Ảnh hưởng của tháng không giống nhau ở mọi vĩ độ: vùng gần xích đạo thường ít biến động theo mùa, còn vùng vĩ độ cao có mùa đông và mùa hè khác biệt rõ. Nếu chỉ đưa riêng `abs_latitude` và các feature tháng, một mô hình tuyến tính khó tự biểu diễn quan hệ này. Do đó, ba feature tương tác được đề xuất:

- `abslat_x_month_sin = abs_latitude × climatic_month_sin`;
- `abslat_x_month_cos = abs_latitude × climatic_month_cos`;
- `abslat_x_years = abs_latitude × years_since_start`.

Hai tương tác đầu cho phép biên độ mùa thay đổi theo khoảng cách tới xích đạo. Tương tác thứ ba kiểm tra xem xu hướng dài hạn có khác nhau theo vĩ độ hay không.

Các feature này đặc biệt hữu ích cho mô hình tuyến tính. Mô hình cây có thể tự học một phần tương tác, nhưng các biến vẫn được giữ ở danh sách ứng viên để đánh giá bằng validation thay vì kết luận trước.

#### 4.1.4. Feature khí hậu chuẩn theo địa điểm

Một thành phố có nền nhiệt riêng và mỗi tháng tại thành phố đó cũng có mức nhiệt điển hình riêng. Vì vậy, notebook xây dựng khóa `location_key` từ `city`, `country`, `latitude` và `longitude`, rồi tạo các thống kê:

- `loc_month_climatology`: nhiệt độ kỳ vọng của đúng địa điểm trong đúng tháng khí hậu;
- `loc_mean_temperature`: nhiệt độ trung bình dài hạn của địa điểm;
- `loc_temperature_std`: độ biến động nhiệt độ lịch sử của địa điểm;
- `loc_climatology_imputed`: cờ cho biết giá trị climatology phải dùng phương án thay thế do nhóm địa điểm–tháng chưa đủ lịch sử.

`location_key` có cả tọa độ vì EDA cho thấy tên thành phố không luôn là định danh duy nhất. Việc dùng khóa ghép tránh gộp nhầm các địa điểm trùng tên.

Đây là nhóm nhạy cảm với leakage vì các thống kê được tính từ chính biến mục tiêu. Quy tắc áp dụng là:

1. Với dòng thuộc tập fit, climatology sử dụng thống kê **leave-one-out** để dòng hiện tại không đóng góp vào feature của chính nó.
2. Với validation và test, toàn bộ bảng thống kê chỉ được fit từ giai đoạn huấn luyện rồi cố định khi biến đổi dữ liệu về sau.
3. Nếu không có đủ lịch sử cho một địa điểm–tháng, notebook lần lượt dùng mức tổng quát hơn và đánh dấu bằng `loc_climatology_imputed`.
4. Các bảng thống kê đã fit được lưu riêng để pipeline dự báo có thể tái sử dụng đúng tham số, không tính lại trên dữ liệu tương lai.

Vì vậy, nhóm này có loại khả dụng là **fit statistic**: không biết sẵn như tọa độ, nhưng có thể dùng hợp lệ sau khi học hoàn toàn từ tập fit.

#### 4.1.5. Feature lịch sử, độ trễ và cửa sổ trượt

Nhiệt độ là chuỗi thời gian có tính liên tục và mùa vụ mạnh. Để dự báo tháng kế tiếp, notebook khai thác các quan sát đã xảy ra:

- `temp_lag_1`: nhiệt độ của tháng ngay trước tháng cần dự báo, phản ánh quán tính ngắn hạn;
- `temp_lag_12`: nhiệt độ của đúng tháng đó một năm trước, phản ánh mùa vụ theo năm;
- `temp_roll_mean_12`: trung bình các quan sát hợp lệ trong 12 tháng trước, mô tả nền nhiệt gần đây;
- `temp_roll_std_12`: độ lệch chuẩn của 12 tháng trước, mô tả mức biến động gần đây;
- `temp_anomaly_lag_12`: chênh lệch giữa nhiệt độ một năm trước và climatology tương ứng, cho biết năm trước nóng/lạnh hơn mức chuẩn bao nhiêu.

Độ trễ được ghép bằng `location_key` và chỉ số tháng liên tục, không dùng `groupby().shift()` một cách mù quáng. Lý do là dữ liệu có thể thiếu một số tháng; dịch theo vị trí dòng có thể lấy nhầm quan sát cách hai hoặc nhiều tháng và gắn nhãn là `lag_1`.

Mọi cửa sổ trượt đều được dịch lùi trước khi tính toán để loại tháng hiện tại. Giá trị thiếu ở đầu chuỗi hoặc sau khoảng trống thời gian là kết quả hợp lệ của việc chưa có đủ lịch sử, không phải lý do để lấy dữ liệu tương lai điền ngược.

Nhóm này chỉ phù hợp với kịch bản đã chốt là **one-step-ahead**: khi dự báo tháng \(t\), dữ liệu đến tháng \(t-1\) được giả định đã quan sát. Nếu chuyển sang dự báo nhiều tháng liên tiếp, chiến lược tạo lag phải được thiết kế lại.

#### 4.1.6. Feature bối cảnh khí hậu toàn cầu

Nhiệt độ thành phố chịu tác động của điều kiện địa phương nhưng cũng có thể đồng biến với nền nhiệt toàn cầu. Hai feature được đề xuất là:

- `land_temperature_lag_1`: nhiệt độ đất liền toàn cầu ở tháng trước;
- `land_anomaly_lag_1`: mức chênh của nhiệt độ toàn cầu tháng trước so với đường chuẩn toàn cầu được fit từ dữ liệu huấn luyện.

Notebook không sử dụng `land_average_temperature` của **cùng tháng mục tiêu**, vì trong dự báo thực tế giá trị đó có thể chưa được công bố. Dịch lùi một tháng giúp bảo toàn trật tự thời gian và làm rõ giả định về khả năng cung cấp dữ liệu.

Nhóm này có thể bổ sung tín hiệu xu hướng chung mà các lag cục bộ chưa biểu diễn hết. Dù vậy, nó có thể tương quan mạnh với `years_since_start`, nên Notebook 05 chỉ ghi nhận đa cộng tuyến; giá trị dự báo bổ sung phải được Notebook 06 kiểm tra bằng validation.

#### 4.1.7. Feature phản ánh chất lượng phép đo

`city_average_temperature_uncertainty` mô tả độ bất định của phép đo nhiệt độ. Giá trị bất định cao có thể gắn với dữ liệu lịch sử cũ, mạng lưới quan trắc thưa hoặc chất lượng ước lượng thấp hơn.

Để không phụ thuộc vào thông tin của tháng đang dự báo, notebook chỉ tạo `city_uncertainty_lag_1`, tức độ bất định của tháng trước. Feature này có thể giúp mô hình điều chỉnh mức tin cậy đối với lịch sử gần nhất.

Tuy nhiên, độ bất định thường thay đổi theo thời kỳ và có thể trùng tín hiệu với `years_since_start`. Vì vậy, Notebook 05 bàn giao nó như một ứng viên và ghi nhận tương quan giữa các feature; Notebook 06 mới đánh giá hiệu quả validation trước khi quyết định sử dụng.

#### 4.1.8. Feature bị loại trước và ranh giới lựa chọn mô hình

Một số cột bị loại ngay tại Notebook 05 vì không thể là đầu vào hợp lệ cho dự báo:

- nhiệt độ thành phố của chính tháng mục tiêu;
- nhiệt độ toàn cầu, quốc gia hoặc thành phố lớn của cùng tháng mục tiêu;
- độ bất định của phép đo trong cùng tháng mục tiêu;
- anomaly hoặc thống kê sử dụng target của chính dòng mà không có leave-one-out;
- thống kê nhóm được fit trên validation/test hoặc toàn bộ dữ liệu trước khi chia thời gian;
- chuỗi `city_name`, `country_name` đưa trực tiếp vào ma trận số.

Đây là **loại theo ngữ nghĩa và data contract**, không phải lựa chọn dựa trên kết quả mô hình. Sau bước này, Notebook 05 bàn giao toàn bộ feature còn lại.

Notebook 06 chịu trách nhiệm:

1. Fit preprocessing cần thiết chỉ trên train.
2. Huấn luyện Linear Regression, Random Forest và XGBoost.
3. Đánh giá bằng validation theo cùng một bộ metric.
4. Tính feature importance phù hợp với từng mô hình, có thể bổ sung permutation importance hoặc SHAP.
5. Thực hiện ablation/feature selection nếu cần và chỉ mở test sau khi pipeline đã chốt.

Như vậy, một feature xuất hiện trong dữ liệu bàn giao có nghĩa là nó **an toàn để thử nghiệm**, không có nghĩa là Notebook 05 đã chứng minh feature đó quan trọng.

### 4.2. Nguyên tắc chống rò rỉ dữ liệu

Rò rỉ dữ liệu là việc dùng thông tin không có tại thời điểm dự báo hoặc để dữ liệu tương lai tác động ngược vào quá trình fit. Notebook áp dụng năm nguyên tắc:

1. Không dùng nhiệt độ hay uncertainty của chính tháng đang dự báo.
2. Chia dữ liệu theo thời gian thành feature-fit, validation và test; Notebook 05 chỉ gắn nhãn, còn Notebook 06 phải khóa test khỏi mọi quyết định mô hình.
3. Target statistics dùng leave-one-out cho chính các dòng feature-fit; validation/test chỉ dùng thống kê cố định từ feature-fit.
4. Lag và rolling tra cứu đúng các tháng trước bằng `location_key - k`.
5. Feature suy ra trực tiếp từ target chỉ dùng phân tích và không xuất sang Notebook 06.

Mục 6.4 kiểm tra tự động các điều kiện quan trọng này.

### 4.3. Khai báo các giai đoạn thời gian và sổ đăng ký Feature

Ba mask thời gian có vai trò riêng:

- `feature_fit_mask`: dữ liệu đến 1983 để học target statistics an toàn.
- `validation_period_mask`: dữ liệu 1984–1993, được bàn giao làm validation cho Notebook 06.
- `test_period_mask`: dữ liệu 1994–2013, chỉ được Notebook 06 mở sau khi pipeline đã chốt.

Notebook 05 không dùng validation/test để tính importance hoặc lựa chọn feature. Hàm `add_feature` chỉ ghi feature vào DataFrame và đăng ký metadata. Trường `availability` cho biết điều kiện dùng lúc dự báo:

- `static`: suy ra từ địa điểm và ngày dự báo.
- `fit_stat`: cần bảng thống kê học từ feature-fit và phải lưu cùng mô hình.
- `history`: cần dữ liệu các tháng trước của đúng địa điểm hoặc chuỗi toàn cầu.

In [ ]:
feature_fit_mask = feature_df['year'] <= FEATURE_FIT_CUTOFF_YEAR
validation_period_mask = feature_df['year'].between(
    VALIDATION_START_YEAR, TRAIN_CUTOFF_YEAR
)
test_period_mask = feature_df['year'] > TRAIN_CUTOFF_YEAR
train_period_mask = feature_fit_mask | validation_period_mask

if (feature_fit_mask & validation_period_mask).any():
    raise RuntimeError('Giai đoạn feature-fit và validation bị chồng lấn.')
if (train_period_mask & test_period_mask).any():
    raise RuntimeError('Giai đoạn train và test bị chồng lấn.')
if not (feature_fit_mask | validation_period_mask | test_period_mask).all():
    raise RuntimeError('Có dòng chưa được gán vào fit/validation/test.')

FEATURE_CATALOG: dict[str, dict] = {}


def add_feature(
    name: str,
    values,
    group: str,
    description: str,
    availability: str = 'static',
) -> None:
    """Ghi một đặc trưng vào feature_df và đăng ký metadata của nó."""
    series = pd.Series(np.asarray(values), index=feature_df.index)
    if series.dtype == bool:
        series = series.astype('int8')
    elif series.dtype.kind == 'f':
        series = series.astype('float32')
    elif series.dtype.kind in 'iu':
        max_abs = series.abs().max()
        series = series.astype('int32' if max_abs > 32_000 else 'int16')
    feature_df[name] = series
    FEATURE_CATALOG[name] = {
        'group': group,
        'description': description,
        'availability': availability,
        'dtype': str(series.dtype),
    }


split_summary = pd.DataFrame(
    {
        'Giai đoạn': ['Feature fit', 'Validation', 'Test'],
        'Khoảng năm': [
            f'<= {FEATURE_FIT_CUTOFF_YEAR}',
            f'{VALIDATION_START_YEAR}–{TRAIN_CUTOFF_YEAR}',
            f'> {TRAIN_CUTOFF_YEAR}',
        ],
        'Số dòng': [
            int(feature_fit_mask.sum()),
            int(validation_period_mask.sum()),
            int(test_period_mask.sum()),
        ],
    }
)
split_summary['Tỷ lệ'] = split_summary['Số dòng'] / len(feature_df)
display(split_summary)

Tỷ lệ dòng train in ra thường cao hơn tỷ lệ số năm, vì Notebook 04 đã chỉ ra độ phủ dữ liệu tăng dần theo thời gian (năm 1863 chỉ có 28.650 dòng). Nếu tỷ lệ test quá nhỏ so với nhu cầu đánh giá, hãy hạ `TRAIN_CUTOFF_YEAR` ở Mục 2.3 và chạy lại từ đây.

## 5. Xây dựng Feature

### 5.1. Khóa vị trí và chỉ số tháng liên tục

Hai khóa kỹ thuật này là nền tảng cho toàn bộ các nhóm đặc trưng phía sau:

- `location_id` — mã số nguyên của một vị trí, ghép từ `city_name + country_name + latitude + longitude`. Notebook 04 đã chỉ ra `city_name` một mình không định danh duy nhất (Springfield, Worcester xuất hiện ở nhiều nơi), nên nếu nhóm theo tên thành phố sẽ trộn lẫn các thành phố khác nhau.
- `month_index` — số tháng liên tục kể từ năm 0, tính bằng `year * 12 + month`. Bảng sạch **có khoảng trống tháng** (Notebook 03 đã loại 58.727 dòng thiếu target), nên `shift(12)` theo số dòng sẽ lấy sai mốc thời gian. Tra cứu theo `month_index - 12` luôn cho đúng cùng tháng năm trước.

Hai khóa được ghép thành một khóa `int64` duy nhất: `location_key = location_id * 30000 + month_index`. Tra cứu trên một chỉ mục `int64` nhanh và nhẹ hơn nhiều so với `MultiIndex` hai tầng trên 5,58 triệu dòng.

In [ ]:
feature_df = feature_df.sort_values(
    LOCATION_KEY_COLUMNS + ['observation_date'], kind='mergesort', ignore_index=True
)

# sort(..., ignore_index=True) thay đổi thứ tự dòng, nên phải tạo lại các mask
# thời gian để nhãn split tiếp tục khớp đúng từng quan sát.
feature_fit_mask = feature_df['year'] <= FEATURE_FIT_CUTOFF_YEAR
validation_period_mask = feature_df['year'].between(
    VALIDATION_START_YEAR, TRAIN_CUTOFF_YEAR
)
test_period_mask = feature_df['year'] > TRAIN_CUTOFF_YEAR
train_period_mask = feature_fit_mask | validation_period_mask

feature_df['location_id'] = (
    feature_df.groupby(LOCATION_KEY_COLUMNS, observed=True, sort=False)
    .ngroup()
    .astype('int32')
)
feature_df['month_index'] = (
    feature_df['year'].astype('int32') * 12 + feature_df['month'].astype('int32')
)

# Điều kiện để phép cộng dồn khóa không bị tràn sang bậc location_id.
MONTH_INDEX_SCALE = 30_000
if feature_df['month_index'].max() >= MONTH_INDEX_SCALE:
    raise RuntimeError('month_index vượt quá MONTH_INDEX_SCALE; hãy tăng hằng số này.')
if feature_df['month_index'].min() < 12:
    raise RuntimeError('month_index quá nhỏ để tra cứu lag 12 tháng.')

feature_df['location_key'] = (
    feature_df['location_id'].astype('int64') * MONTH_INDEX_SCALE
    + feature_df['month_index'].astype('int64')
)
if not feature_df['location_key'].is_unique:
    raise RuntimeError('location_key không duy nhất; đặc trưng trễ sẽ tra cứu sai.')

n_locations = int(feature_df['location_id'].nunique())
print(f'Số vị trí               : {n_locations:,}')
print(f'Số dòng                 : {len(feature_df):,}')
print(f'Số tháng trung bình/vị trí: {len(feature_df) / n_locations:,.0f}')
print(f'location_key duy nhất   : PASS')

`location_key duy nhất : PASS` là điều kiện tiên quyết cho Mục 5.6: nó bảo đảm mỗi cặp (vị trí, tháng) chỉ có một giá trị nhiệt độ, nên phép tra cứu lag trả về đúng một kết quả. Số tháng trung bình trên mỗi vị trí cho biết chuỗi thời gian đủ dài để tính trung bình trượt 12 tháng hay không.

### 5.2. Nhóm 1 — Đặc trưng thời gian

Nhóm này giải quyết ba hạn chế của các cột lịch thô.

**Mã hóa tuần hoàn.** Thay `month` bằng cặp `sin`/`cos` trên chu kỳ 12 tháng, để tháng 12 và tháng 1 nằm cạnh nhau trong không gian đặc trưng. Cặp này phải luôn được giữ cùng nhau — một mình `sin` không phân biệt được các cặp như tháng 1–5, tháng 7–11 hoặc tháng 6–12 — nên cả hai luôn được tạo và bàn giao cùng nhau; Notebook 06 không nên loại riêng một thành phần mà không kiểm chứng.

**Hiệu chỉnh bán cầu.** Ở Nam bán cầu tháng 1 là giữa hè còn tháng 7 là giữa đông, ngược pha hoàn toàn với Bắc bán cầu. Dữ liệu có vĩ độ từ −52.24 đến 69.92 nên nếu dùng chung một phép mã hóa cho cả hai bán cầu, tín hiệu mùa vụ của hai nhóm sẽ triệt tiêu nhau. `climatic_month` dịch 6 tháng cho các vị trí Nam bán cầu để đưa cả hai về cùng một pha khí hậu.

**Biến trôi thời gian.** `years_since_start` đếm số năm kể từ mốc đầu dữ liệu, cho mô hình một trục tuyến tính để hấp thụ xu hướng xấp xỉ +0,027 °C/năm.

In [ ]:
month_values = feature_df['month'].to_numpy(dtype='float32')
is_north = (feature_df['latitude'] >= 0).to_numpy()

# Nam bán cầu lệch pha 6 tháng: tháng 1 (giữa hè) tương ứng pha tháng 7 ở Bắc bán cầu.
climatic_month = np.where(is_north, month_values, ((month_values + 5) % 12) + 1)
feature_df['climatic_month'] = climatic_month.astype('int8')

TWO_PI_OVER_12 = 2 * np.pi / 12
add_feature('month_sin', np.sin(TWO_PI_OVER_12 * month_values), 'Thời gian',
            'Mã hóa tuần hoàn tháng theo lịch (sin).')
add_feature('month_cos', np.cos(TWO_PI_OVER_12 * month_values), 'Thời gian',
            'Mã hóa tuần hoàn tháng theo lịch (cos).')
add_feature('climatic_month_sin', np.sin(TWO_PI_OVER_12 * climatic_month), 'Thời gian',
            'Mã hóa tháng đã hiệu chỉnh bán cầu (sin).')
add_feature('climatic_month_cos', np.cos(TWO_PI_OVER_12 * climatic_month), 'Thời gian',
            'Mã hóa tháng đã hiệu chỉnh bán cầu (cos).')

start_year = int(feature_df['year'].min())
add_feature(
    'years_since_start',
    feature_df['year'].astype('int32') - start_year,
    'Thời gian',
    f'Số năm kể từ {start_year}; biểu diễn xu hướng dài hạn.',
)

# Nhãn mùa chỉ dùng cho phân tích, không đưa trực tiếp vào feature đầu ra.
feature_df['climatic_season'] = pd.cut(
    feature_df['climatic_month'],
    bins=[0, 2, 5, 8, 11, 12],
    labels=['Đông', 'Xuân', 'Hạ', 'Thu', 'Đông'],
    ordered=False,
)

print(f'Mốc năm đầu tiên: {start_year}')
display(feature_df[
    ['month', 'latitude', 'climatic_month', 'climatic_season',
     'month_sin', 'climatic_month_sin', 'years_since_start']
].head())

Cách kiểm tra nhanh cell này: với một dòng ở Bắc bán cầu (`latitude >= 0`), `climatic_month` phải bằng `month`; với một dòng ở Nam bán cầu, `climatic_month` phải lệch đúng 6 tháng (tháng 1 → 7, tháng 12 → 6). Cột `climatic_season` là nhãn phục vụ phân tích, không nằm trong danh sách đặc trưng.

### 5.3. Nhóm 2 — Đặc trưng địa lý

Notebook 04 xác định `latitude` là driver mạnh nhất trong các biến thô (`r = -0.467`) nhưng cũng chỉ ra hạn chế của nó: vì có dấu, một giá trị vĩ độ trộn lẫn hai thông tin khác nhau là **khoảng cách tới xích đạo** (quyết định nền nhiệt) và **bán cầu** (quyết định pha mùa vụ). Nhóm này tách chúng ra:

- `abs_latitude` — khoảng cách tới xích đạo. Đây mới là biến mang quan hệ đơn điệu với nhiệt độ: càng xa xích đạo càng lạnh, bất kể Bắc hay Nam.
- `hemisphere_north` — bán cầu, đã được dùng ở Mục 5.2 để hiệu chỉnh pha mùa vụ.
- `longitude` và `is_major_city` giữ nguyên làm thuộc tính vị trí.

Notebook **không** tạo thêm `distance_to_equator_km` vì nó chỉ là `abs_latitude × 111` — cộng tuyến tuyệt đối, không thêm thông tin nào.

In [ ]:
latitude_values = feature_df['latitude'].to_numpy(dtype='float32')
abs_latitude_values = np.abs(latitude_values)

add_feature('latitude', latitude_values, 'Địa lý',
            'Vĩ độ có dấu; âm là Nam bán cầu.')
add_feature('abs_latitude', abs_latitude_values, 'Địa lý',
            'Khoảng cách tới xích đạo, driver nền nhiệt của vị trí.')
add_feature('hemisphere_north', is_north, 'Địa lý',
            'Vị trí thuộc Bắc bán cầu (1) hay Nam bán cầu (0).')
add_feature('longitude', feature_df['longitude'].to_numpy(dtype='float32'), 'Địa lý',
            'Kinh độ của thành phố.')
add_feature('is_major_city', feature_df['is_major_city'].to_numpy(), 'Địa lý',
            'Thành phố thuộc nhóm Major City của bộ dữ liệu gốc.')

# Vành đai vĩ độ chỉ dùng để phân tích ở Mục 8, không phải đặc trưng.
feature_df['latitude_zone'] = pd.cut(
    feature_df['abs_latitude'],
    bins=[0, 23.5, 35, 55, 90],
    labels=['Nhiệt đới', 'Cận nhiệt', 'Ôn đới', 'Cận cực'],
    include_lowest=True,
)

display(
    feature_df.groupby('latitude_zone', observed=True)
    .agg(
        so_dong=('location_id', 'size'),
        so_vi_tri=('location_id', 'nunique'),
        nhiet_do_tb=(TARGET_COLUMN, 'mean'),
    )
    .rename_axis('Vành đai vĩ độ')
)
print(f"Tỷ lệ dòng ở Bắc bán cầu: {feature_df['hemisphere_north'].mean():.1%}")

Bảng trên là kiểm tra tính hợp lý của `abs_latitude`: nhiệt độ trung bình phải **giảm dần** từ Nhiệt đới đến Cận cực. Nếu thứ tự này bị đảo, rất có thể `abs_latitude` đã bị tính sai dấu.

### 5.4. Nhóm 3 — Đặc trưng tương tác

Notebook 04 phát hiện một điều mà không đặc trưng đơn lẻ nào biểu diễn được: **độ lệch chuẩn theo tháng không đều** — mùa đông 11.9–12.9 °C nhưng mùa hè chỉ 5.2–6.1 °C. Nguyên nhân là biên độ mùa vụ phụ thuộc vĩ độ: thành phố gần xích đạo gần như không có mùa, còn thành phố vĩ độ cao dao động rất mạnh giữa đông và hè.

Quan hệ đó là quan hệ **nhân**, không phải cộng, nên mô hình tuyến tính không thể học từ `abs_latitude` và `climatic_month_sin` riêng lẻ. Hai đặc trưng tương tác đầu tiên cung cấp trực tiếp tích của chúng. Đặc trưng thứ ba cho phép tốc độ ấm lên khác nhau theo vĩ độ.

In [ ]:
climatic_month_sin_values = feature_df['climatic_month_sin'].to_numpy()
climatic_month_cos_values = feature_df['climatic_month_cos'].to_numpy()
years_since_start_values = feature_df['years_since_start'].to_numpy(dtype='float32')

add_feature('abslat_x_month_sin', abs_latitude_values * climatic_month_sin_values,
            'Tương tác',
            'Biên độ mùa vụ tăng theo vĩ độ (thành phần sin).')
add_feature('abslat_x_month_cos', abs_latitude_values * climatic_month_cos_values,
            'Tương tác',
            'Biên độ mùa vụ tăng theo vĩ độ (thành phần cos).')
add_feature('abslat_x_years', abs_latitude_values * years_since_start_values,
            'Tương tác',
            'Cho phép tốc độ ấm lên khác nhau giữa các vành đai vĩ độ.')

# Minh chứng cho ý tưởng: biên độ mùa vụ theo vành đai vĩ độ.
seasonal_check = (
    feature_df.groupby(['latitude_zone', 'climatic_month'], observed=True)[TARGET_COLUMN]
    .mean()
    .unstack('climatic_month')
)
display(
    (seasonal_check.max(axis=1) - seasonal_check.min(axis=1))
    .to_frame('Biên độ mùa vụ (°C)')
    .rename_axis('Vành đai vĩ độ')
)

Bảng biên độ mùa vụ là bằng chứng định lượng cho nhóm tương tác: biên độ phải **tăng dần** từ Nhiệt đới đến Cận cực. Chênh lệch giữa hai đầu càng lớn thì hai đặc trưng `abslat_x_month_*` càng có giá trị, vì đó chính là phần thông tin mà `abs_latitude` và `climatic_month_sin` đứng riêng không thể diễn tả.

### 5.5. Nhóm 4 — Đặc trưng khí hậu theo vị trí

Đây là nhóm mạnh nhưng dễ rò rỉ vì được tổng hợp từ target.

Notebook chỉ học thống kê từ giai đoạn `year <= FEATURE_FIT_CUTOFF_YEAR`. Với chính các dòng thuộc giai đoạn này, giá trị target của dòng được trừ khỏi tổng và số đếm trước khi tính trung bình/độ lệch chuẩn (**leave-one-out**). Validation và test chỉ nhận thống kê cố định từ feature-fit, nên target của chúng không tham gia vào feature.

Nếu vị trí chưa có đủ lịch sử, fallback lần lượt dùng thống kê theo vành đai–tháng và thống kê toàn cục, cũng chỉ từ feature-fit và leave-one-out khi cần.

In [ ]:
# Chỉ giai đoạn feature-fit (<= 1983) được dùng để học target statistics.
fit_slice = feature_df.loc[
    feature_fit_mask,
    ['location_id', 'month', 'latitude_zone', TARGET_COLUMN],
].copy()
fit_slice['_target_squared'] = fit_slice[TARGET_COLUMN] ** 2

target_values = feature_df[TARGET_COLUMN].to_numpy(dtype='float64')
fit_positions = np.flatnonzero(feature_fit_mask.to_numpy())

# --- Trung bình theo (vị trí, tháng) ------------------------------------
loc_month_stats = fit_slice.groupby(
    ['location_id', 'month'], observed=True
)[TARGET_COLUMN].agg(['count', 'sum'])
loc_month_index = pd.MultiIndex.from_arrays(
    [feature_df['location_id'], feature_df['month']]
)
loc_month_count = loc_month_stats['count'].reindex(loc_month_index).to_numpy(dtype='float64')
loc_month_sum = loc_month_stats['sum'].reindex(loc_month_index).to_numpy(dtype='float64')
climatology_values = loc_month_sum / loc_month_count

# Với chính các dòng feature-fit, dùng leave-one-out để target của dòng không
# tham gia vào feature của nó. Validation và test dùng thống kê cố định từ fit.
loo_count = loc_month_count[fit_positions] - 1
loo_sum = loc_month_sum[fit_positions] - target_values[fit_positions]
climatology_values[fit_positions] = np.divide(
    loo_sum,
    loo_count,
    out=np.full_like(loo_sum, np.nan),
    where=loo_count > 0,
)

# --- Fallback theo (vành đai, tháng), rồi theo tháng toàn cục -----------
zone_month_stats = fit_slice.groupby(
    ['latitude_zone', 'month'], observed=True
)[TARGET_COLUMN].agg(['count', 'sum'])
zone_month_index = pd.MultiIndex.from_arrays(
    [feature_df['latitude_zone'], feature_df['month']]
)
zone_count = zone_month_stats['count'].reindex(zone_month_index).to_numpy(dtype='float64')
zone_sum = zone_month_stats['sum'].reindex(zone_month_index).to_numpy(dtype='float64')
zone_values = zone_sum / zone_count
zone_loo_count = zone_count[fit_positions] - 1
zone_loo_sum = zone_sum[fit_positions] - target_values[fit_positions]
zone_values[fit_positions] = np.divide(
    zone_loo_sum,
    zone_loo_count,
    out=np.full_like(zone_loo_sum, np.nan),
    where=zone_loo_count > 0,
)

global_month_stats = fit_slice.groupby('month', observed=True)[TARGET_COLUMN].agg(
    ['count', 'sum']
)
global_count = feature_df['month'].map(global_month_stats['count']).to_numpy(dtype='float64')
global_sum = feature_df['month'].map(global_month_stats['sum']).to_numpy(dtype='float64')
global_values = global_sum / global_count
global_loo_count = global_count[fit_positions] - 1
global_loo_sum = global_sum[fit_positions] - target_values[fit_positions]
global_values[fit_positions] = np.divide(
    global_loo_sum,
    global_loo_count,
    out=np.full_like(global_loo_sum, np.nan),
    where=global_loo_count > 0,
)

imputed_mask = np.isnan(climatology_values)
climatology_values[imputed_mask] = zone_values[imputed_mask]
still_missing = np.isnan(climatology_values)
climatology_values[still_missing] = global_values[still_missing]
if np.isnan(climatology_values).any():
    raise RuntimeError('Không thể tạo climatology an toàn cho một số dòng.')

add_feature(
    'loc_month_climatology', climatology_values, 'Khí hậu theo vị trí',
    'Nhiệt độ bình thường của vị trí theo tháng; leave-one-out ở fit, '
    'cố định từ fit cho validation/test.',
    availability='fit_stat',
)
add_feature(
    'loc_climatology_imputed', imputed_mask, 'Khí hậu theo vị trí',
    'Dòng phải dùng fallback theo vành đai/tháng.',
    availability='fit_stat',
)

# --- Thống kê mức vị trí với leave-one-out ở giai đoạn fit --------------
location_stats = fit_slice.groupby('location_id', observed=True).agg(
    count=(TARGET_COLUMN, 'count'),
    total=(TARGET_COLUMN, 'sum'),
    total_sq=('_target_squared', 'sum'),
)
location_ids = feature_df['location_id']
loc_count = location_ids.map(location_stats['count']).to_numpy(dtype='float64')
loc_sum = location_ids.map(location_stats['total']).to_numpy(dtype='float64')
loc_sum_sq = location_ids.map(location_stats['total_sq']).to_numpy(dtype='float64')

loc_mean_values = loc_sum / loc_count
loc_variance = (loc_sum_sq - (loc_sum ** 2) / loc_count) / (loc_count - 1)
loc_std_values = np.sqrt(np.maximum(loc_variance, 0))

fit_loc_count = loc_count[fit_positions] - 1
fit_loc_sum = loc_sum[fit_positions] - target_values[fit_positions]
fit_loc_sum_sq = loc_sum_sq[fit_positions] - target_values[fit_positions] ** 2
fit_loc_mean = np.divide(
    fit_loc_sum,
    fit_loc_count,
    out=np.full_like(fit_loc_sum, np.nan),
    where=fit_loc_count > 0,
)
fit_loc_variance = np.divide(
    fit_loc_sum_sq - np.divide(
        fit_loc_sum ** 2,
        fit_loc_count,
        out=np.zeros_like(fit_loc_sum),
        where=fit_loc_count > 0,
    ),
    fit_loc_count - 1,
    out=np.full_like(fit_loc_sum, np.nan),
    where=fit_loc_count > 1,
)
loc_mean_values[fit_positions] = fit_loc_mean
loc_std_values[fit_positions] = np.sqrt(np.maximum(fit_loc_variance, 0))

add_feature(
    'loc_mean_temperature', loc_mean_values, 'Khí hậu theo vị trí',
    'Nhiệt độ trung bình lịch sử của vị trí, chống self-target leakage.',
    availability='fit_stat',
)
add_feature(
    'loc_temperature_std', loc_std_values, 'Khí hậu theo vị trí',
    'Độ lệch chuẩn lịch sử của vị trí, chống self-target leakage.',
    availability='fit_stat',
)

print(f'Số cặp (vị trí, tháng) trong feature-fit: {len(loc_month_stats):,}')
print(f'Số dòng dùng fallback: {int(imputed_mask.sum()):,} ({imputed_mask.mean():.3%})')
display(feature_df[
    ['loc_month_climatology', 'loc_mean_temperature', 'loc_temperature_std']
].describe())

Bảng kết quả cần kiểm tra tỷ lệ fallback và tỷ lệ thiếu. `loc_month_climatology`, `loc_mean_temperature` và `loc_temperature_std` không được sử dụng target của chính dòng khi tạo feature cho giai đoạn fit; đây là khác biệt quan trọng so với target encoding trực tiếp.

### 5.6. Nhóm 5 — Đặc trưng trễ và trượt

Nhóm này khai thác bản chất chuỗi thời gian của dữ liệu. Điểm cần cẩn thận nhất là **cách tra cứu quá khứ**.

Cách làm sai phổ biến là `groupby('location_id')[target].shift(12)`: nó lùi 12 **dòng**, không phải 12 **tháng**. Bảng sạch có khoảng trống tháng (Notebook 03 đã loại 58.727 dòng thiếu target), nên với một thành phố có tháng bị khuyết, `shift(12)` sẽ trả về giá trị của một mốc thời gian khác.

Cách làm ở đây là tra cứu theo lịch: giá trị lag `k` của một dòng là target tại `location_key - k`, tức đúng vị trí đó và đúng `k` tháng trước. Nếu tháng đó không tồn tại, kết quả là `NaN` — đúng về mặt ngữ nghĩa, thay vì lấy sai mốc.

Toàn bộ 12 lag được tính một lần vào một bảng tạm, dùng để suy ra bốn đặc trưng, rồi giải phóng bộ nhớ ngay.

In [ ]:
target_lookup = pd.Series(
    feature_df[TARGET_COLUMN].to_numpy(),
    index=feature_df['location_key'].to_numpy(),
)
base_keys = feature_df['location_key'].to_numpy()

# Bảng tạm chứa 12 lag theo lịch; chỉ tồn tại trong cell này.
lag_frame = pd.DataFrame(
    {
        f'lag_{lag}': target_lookup.reindex(base_keys - lag).to_numpy(dtype='float32')
        for lag in range(1, 13)
    }
)

MIN_HISTORY_MONTHS = 6
history_count = lag_frame.notna().sum(axis=1)
enough_history = history_count >= MIN_HISTORY_MONTHS

rolling_mean = lag_frame.mean(axis=1).where(enough_history)
rolling_std = lag_frame.std(axis=1).where(enough_history)

add_feature('temp_lag_1', lag_frame['lag_1'].to_numpy(), 'Trễ / Trượt',
            'Nhiệt độ của chính thành phố đó một tháng trước.',
            availability='history')
add_feature('temp_lag_12', lag_frame['lag_12'].to_numpy(), 'Trễ / Trượt',
            'Nhiệt độ cùng tháng của năm trước.',
            availability='history')
add_feature('temp_roll_mean_12', rolling_mean.to_numpy(), 'Trễ / Trượt',
            'Trung bình nhiệt độ 12 tháng trước đó (xấp xỉ nền nhiệt năm gần nhất).',
            availability='history')
add_feature('temp_roll_std_12', rolling_std.to_numpy(), 'Trễ / Trượt',
            'Độ lệch chuẩn nhiệt độ 12 tháng trước đó.',
            availability='history')
add_feature(
    'temp_anomaly_lag_12',
    lag_frame['lag_12'].to_numpy() - feature_df['loc_month_climatology'].to_numpy(),
    'Trễ / Trượt',
    'Cùng tháng năm trước lệch bao nhiêu so với trung bình khí hậu của vị trí.',
    availability='history',
)

lag_coverage = pd.DataFrame(
    {
        'Số dòng có giá trị': [
            int(feature_df['temp_lag_1'].notna().sum()),
            int(feature_df['temp_lag_12'].notna().sum()),
            int(feature_df['temp_roll_mean_12'].notna().sum()),
        ],
        'Tỷ lệ': [
            feature_df['temp_lag_1'].notna().mean(),
            feature_df['temp_lag_12'].notna().mean(),
            feature_df['temp_roll_mean_12'].notna().mean(),
        ],
    },
    index=['temp_lag_1', 'temp_lag_12', 'temp_roll_mean_12'],
)
display(lag_coverage)

del lag_frame, target_lookup, rolling_mean, rolling_std
gc.collect()
print('Đã giải phóng bảng lag tạm.')

Tỷ lệ dòng có giá trị nên ở mức cao (khoảng 98–99%) chứ không phải 100%: các dòng đầu chuỗi của mỗi thành phố không có quá khứ để tra cứu, và các dòng liền sau một tháng khuyết cũng vậy. Đây là kết quả **đúng** — `NaN` ở đây trung thực hơn việc lấp bằng một giá trị sai mốc thời gian.

Notebook 06 có hai lựa chọn với các dòng này: loại chúng khỏi tập huấn luyện, hoặc dùng mô hình chấp nhận giá trị thiếu. Nhóm đặc trưng này có `availability = history`, nghĩa là mô hình dùng chúng chỉ dự báo được khi đã có chuỗi quá khứ của thành phố — điều này sẽ được ghi rõ trong metadata ở Mục 9.3.

### 5.7. Nhóm 6 — Đặc trưng bối cảnh toàn cầu

`land_average_temperature` của chính tháng đang dự báo không chắc đã có sẵn, nên không được dùng trực tiếp. Mục này chỉ tạo hai feature lịch sử:

- `land_temperature_lag_1`: nhiệt độ đất toàn cầu tháng trước.
- `land_anomaly_lag_1`: giá trị tháng trước lệch khỏi mức bình thường của tháng tương ứng, với baseline học từ feature-fit.

Hai feature có `availability = history`, phù hợp kịch bản dự báo một tháng kế tiếp.

In [ ]:
# Mỗi tháng chỉ có một chỉ số toàn cầu; dùng tháng trước để phù hợp kịch bản
# dự báo một bước, không dùng giá trị của chính tháng đang dự báo.
global_by_month_index = (
    feature_df.groupby('month_index', observed=True)['land_average_temperature']
    .mean()
)
land_lag_1 = global_by_month_index.reindex(
    feature_df['month_index'].to_numpy() - 1
).to_numpy(dtype='float32')

global_fit = (
    feature_df.loc[feature_fit_mask, ['month_index', 'month', 'land_average_temperature']]
    .drop_duplicates('month_index')
)
global_month_baseline = global_fit.groupby('month', observed=True)[
    'land_average_temperature'
].mean()
previous_month = ((feature_df['month'].astype('int16') - 2) % 12) + 1
previous_month_baseline = previous_month.map(global_month_baseline).to_numpy(dtype='float32')

add_feature(
    'land_temperature_lag_1', land_lag_1, 'Bối cảnh toàn cầu',
    'Nhiệt độ đất toàn cầu của tháng trước.',
    availability='history',
)
add_feature(
    'land_anomaly_lag_1', land_lag_1 - previous_month_baseline,
    'Bối cảnh toàn cầu',
    'Chỉ số toàn cầu tháng trước lệch khỏi khí hậu tháng tương ứng.',
    availability='history',
)

display(global_month_baseline.to_frame('Nhiệt độ toàn cầu trung bình (°C)').T)
print(f"Số dòng thiếu land lag 1: {int(feature_df['land_temperature_lag_1'].isna().sum()):,}")

Bảng baseline theo tháng chỉ được học từ giai đoạn feature-fit. Số dòng thiếu của `land_temperature_lag_1` chủ yếu xuất hiện ở đầu chuỗi, khi tháng trước chưa có trong phạm vi dữ liệu.

### 5.8. Nhóm 7 — Đặc trưng chất lượng dữ liệu

Uncertainty của phép đo tháng hiện tại chỉ xuất hiện cùng target, nên không phải feature hợp lệ cho dự báo tương lai. Notebook thay nó bằng `city_uncertainty_lag_1` — độ bất định của tháng trước, là thông tin lịch sử đã biết tại thời điểm dự báo.

In [ ]:
# Uncertainty của tháng hiện tại chỉ xuất hiện cùng phép đo target, nên không
# dùng trực tiếp. Chỉ uncertainty của tháng trước là thông tin lịch sử hợp lệ.
uncertainty_lookup = pd.Series(
    feature_df['city_average_temperature_uncertainty'].to_numpy(dtype='float32'),
    index=feature_df['location_key'].to_numpy(),
)
uncertainty_lag_1 = uncertainty_lookup.reindex(
    feature_df['location_key'].to_numpy() - 1
).to_numpy(dtype='float32')

add_feature(
    'city_uncertainty_lag_1', uncertainty_lag_1, 'Chất lượng dữ liệu',
    'Độ bất định của phép đo thành phố ở tháng trước.',
    availability='history',
)

uncertainty_by_decade = (
    feature_df.groupby('decade', observed=True)['city_average_temperature_uncertainty']
    .mean()
    .to_frame('Độ bất định trung bình (°C)')
)
display(uncertainty_by_decade.T)
print(
    'Số dòng thiếu uncertainty lag 1: '
    f"{int(feature_df['city_uncertainty_lag_1'].isna().sum()):,}"
)

Bảng theo thập kỷ vẫn dùng uncertainty gốc để mô tả chất lượng đo lường. Candidate feature đưa sang bước lựa chọn chỉ là phiên bản lag một tháng; uncertainty cùng tháng không được xuất.

### 5.9. Nhóm 8 — Mã hóa biến phân loại quốc gia

Ba mô hình dự kiến ở Notebook 06 — Linear Regression, Random Forest và XGBoost — đều cần ma trận đầu vào dạng số. Vì vậy, `country_name` không thể được đưa trực tiếp vào `X` dưới dạng chuỗi. Mục này sử dụng **One-Hot Encoding** để biểu diễn quốc gia bằng các cột nhị phân:

- Dòng thuộc quốc gia tương ứng nhận giá trị `1`; các dòng còn lại nhận `0`.
- Danh sách quốc gia chỉ được fit từ giai đoạn `year <= FEATURE_FIT_CUTOFF_YEAR`, sau đó giữ cố định khi biến đổi validation và test.
- Một quốc gia được bỏ làm **nhóm tham chiếu** (`drop='first'`). Với 50 quốc gia, notebook tạo 49 cột thay vì 50 cột, tránh quan hệ cộng tuyến hoàn hảo giữa toàn bộ dummy và intercept của Linear Regression.
- Quốc gia chưa xuất hiện ở feature-fit sẽ có toàn bộ cột one-hot bằng `0`, tương đương chính sách `handle_unknown='ignore'`. Notebook đồng thời báo số dòng unknown để người thực hiện phát hiện trường hợp này.
- Cột `country_name` gốc vẫn được giữ trong bảng đầu ra để tra cứu, phân tích sai số theo quốc gia và phục vụ giao diện; nó không nằm trong `feature_names` dưới dạng chuỗi.

**Tại sao không one-hot `city_name`?** Dữ liệu có hơn 3.000 thành phố. Mã hóa city sẽ tạo hơn 3.000 cột rất thưa; ở hơn 5,5 triệu dòng, biểu diễn dense sẽ tiêu tốn bộ nhớ rất lớn và còn vượt giới hạn cột thực tế của bảng PostgreSQL đầu ra. Nó cũng làm mô hình dễ ghi nhớ danh tính thành phố thay vì học quy luật khí hậu, trong khi không giúp dự báo cho thành phố chưa xuất hiện trong train. Thông tin địa điểm đã được biểu diễn có ý nghĩa hơn qua `latitude`, `longitude`, feature mùa vụ, lag và climatology theo `location_id`.

Khối one-hot quốc gia được giữ như một nhóm hoàn chỉnh trong danh sách feature. Việc loại rời rạc từng dummy dựa trên tương quan đơn biến sẽ làm thay đổi ý nghĩa của biến phân loại và tạo nhóm tham chiếu không nhất quán.

#### 5.9.1. Xác định danh sách quốc gia và nhóm tham chiếu

Khối đầu tiên chỉ học danh sách category từ giai đoạn feature-fit. Việc khóa danh sách này trước khi xử lý validation/test bảo đảm các tập sau không tự tạo thêm cột và thứ tự đầu vào của mô hình luôn ổn định.

Tên quốc gia được chuẩn hóa thành hậu tố ASCII, chữ thường và dấu cách thay bằng `_` để tên cột dùng an toàn trong CSV, Python và PostgreSQL. Quốc gia đầu tiên theo thứ tự chữ cái được chọn làm nhóm tham chiếu và không tạo dummy riêng.

In [ ]:
def make_country_slug(country_name: str) -> str:
    # Chuyển tên quốc gia thành hậu tố cột ASCII, chữ thường và ổn định.
    normalized = unicodedata.normalize('NFKD', country_name)
    ascii_name = normalized.encode('ascii', errors='ignore').decode('ascii')
    slug = re.sub(r'[^a-z0-9]+', '_', ascii_name.lower()).strip('_')

    # Giới hạn độ dài để tên cột luôn an toàn với PostgreSQL.
    return (slug or 'unknown_country')[:42]


# Chỉ học danh sách category từ feature-fit; validation/test không tạo cột mới.
FIT_COUNTRY_CATEGORIES = sorted(
    str(country)
    for country in feature_df.loc[feature_fit_mask, 'country_name'].dropna().unique()
)
if len(FIT_COUNTRY_CATEGORIES) < 2:
    raise RuntimeError('Cần ít nhất hai quốc gia trong feature-fit để one-hot encoding.')

# Bỏ category đầu tiên để làm nhóm tham chiếu cho Linear Regression.
COUNTRY_REFERENCE_COUNTRY = FIT_COUNTRY_CATEGORIES[0]

print(f'Số quốc gia học từ feature-fit: {len(FIT_COUNTRY_CATEGORIES)}')
print(f'Nhóm tham chiếu                : {COUNTRY_REFERENCE_COUNTRY}')

Kết quả cho thấy feature-fit có đủ **50 quốc gia**. `Argentina` đứng đầu theo thứ tự chữ cái nên được chọn làm nhóm tham chiếu. Các dòng của Argentina sẽ có toàn bộ dummy quốc gia bằng 0; đây là quy ước của `drop='first'`, không phải dữ liệu thiếu.

#### 5.9.2. Tạo các cột One-Hot Encoding

Khối này duyệt qua các quốc gia còn lại và tạo một cột nhị phân cho từng quốc gia. Các cột được đăng ký bằng `add_feature()` nên tự động đi vào `FEATURE_CATALOG`, có kiểu `int8` và được ghi kèm metadata.

Toàn bộ khối one-hot được tạo và bàn giao cùng nhau. Notebook 05 không loại rời rạc từng dummy theo tương quan hay importance; nếu Notebook 06 thử feature selection, cần xem country như một nhóm để không làm thay đổi nhóm tham chiếu một cách thiếu nhất quán.

In [ ]:
COUNTRY_OHE_MAPPING = {}
encoded_country_count = np.zeros(len(feature_df), dtype='int8')
used_feature_names = set()

for position, country in enumerate(FIT_COUNTRY_CATEGORIES[1:], start=1):
    base_name = f'country_ohe__{make_country_slug(country)}'
    feature_name = base_name

    # Thêm số thứ tự nếu hai tên quốc gia tạo ra cùng một slug.
    if feature_name in used_feature_names:
        feature_name = f'{base_name}_{position:02d}'
    used_feature_names.add(feature_name)

    # Giữ kiểu bool để add_feature nén cột nhị phân xuống int8.
    indicator = feature_df['country_name'].eq(country).to_numpy(dtype=bool)
    add_feature(
        feature_name,
        indicator,
        'Phân loại quốc gia',
        f'Quốc gia là {country}; nhóm tham chiếu là {COUNTRY_REFERENCE_COUNTRY}.',
        availability='static',
    )

    COUNTRY_OHE_MAPPING[feature_name] = country
    encoded_country_count += indicator

print(f'Số cột one-hot được tạo: {len(COUNTRY_OHE_MAPPING)}')

Notebook đã tạo đúng **49 cột one-hot** từ 50 quốc gia, thỏa mãn công thức `số quốc gia feature-fit − 1`. Các cột được lưu bằng `int8` và chỉ nhận `0/1`, phù hợp để đưa trực tiếp vào Linear Regression, Random Forest và XGBoost.

#### 5.9.3. Kiểm tra tính đúng đắn và quốc gia chưa biết

Khối kiểm tra thực thi data contract của one-hot encoding:

1. Một dòng không được thuộc nhiều hơn một dummy quốc gia.
2. Mọi quốc gia đã biết và không phải nhóm tham chiếu phải có đúng một dummy bằng 1.
3. Nhóm tham chiếu và quốc gia chưa có trong feature-fit phải có toàn bộ dummy bằng 0.

Quốc gia mới được xử lý tương đương `handle_unknown='ignore'`. Bảng tổng hợp theo split giúp phát hiện trường hợp validation hoặc test xuất hiện category mới.

In [ ]:
# Category ngoài danh sách fit được xem là unknown và không tạo cột mới.
known_country_mask = feature_df['country_name'].isin(
    FIT_COUNTRY_CATEGORIES
).to_numpy()
reference_country_mask = feature_df['country_name'].eq(
    COUNTRY_REFERENCE_COUNTRY
).to_numpy()
unknown_country_mask = ~known_country_mask
expected_one_mask = known_country_mask & ~reference_country_mask

if (encoded_country_count > 1).any():
    raise RuntimeError('Một dòng được gán vào nhiều hơn một cột one-hot country.')
if not np.array_equal(encoded_country_count == 1, expected_one_mask):
    raise RuntimeError('Kết quả one-hot country không khớp category và nhóm tham chiếu.')

unknown_by_split = pd.DataFrame(
    {
        'Split': ['Feature fit', 'Validation', 'Test'],
        'Số dòng country unknown': [
            int((unknown_country_mask & feature_fit_mask.to_numpy()).sum()),
            int((unknown_country_mask & validation_period_mask.to_numpy()).sum()),
            int((unknown_country_mask & test_period_mask.to_numpy()).sum()),
        ],
    }
)
display(unknown_by_split)
print('PASS: mỗi dòng có mã hóa country đúng với data contract.')

Cả `Feature fit`, `Validation` và `Test` đều có **0 dòng country unknown**. Như vậy, mọi quốc gia ở các giai đoạn sau đều đã xuất hiện trong feature-fit và được ánh xạ vào đúng hệ cột one-hot đã khóa. Thông báo `PASS` xác nhận không có dòng được gán sai hoặc đồng thời thuộc nhiều dummy quốc gia.

#### 5.9.4. Lập bảng mapping để bàn giao

Bảng mapping nối tên quốc gia dễ đọc với tên feature thực tế. Mapping này còn được ghi vào `feature_metadata.json`, giúp Notebook 06 và ứng dụng dự báo tái tạo đúng cột, đúng tên và đúng nhóm tham chiếu.

In [ ]:
country_mapping_table = pd.DataFrame(
    [
        {
            'Quốc gia': COUNTRY_REFERENCE_COUNTRY,
            'Feature': '(nhóm tham chiếu — toàn bộ dummy bằng 0)',
        }
    ]
    + [
        {'Quốc gia': country, 'Feature': feature_name}
        for feature_name, country in COUNTRY_OHE_MAPPING.items()
    ]
)

display(country_mapping_table)

Bảng mapping có **50 dòng**: một dòng cho nhóm tham chiếu `Argentina` và 49 dòng tương ứng với các cột `country_ohe__*`. Mapping này phải được lưu nguyên vẹn trong `feature_metadata.json` để Notebook 06 và ứng dụng suy luận tạo đúng tên, thứ tự cột. `country_name` gốc vẫn được giữ để tra cứu và phân tích, nhưng không được đưa trực tiếp vào ma trận `X`.

### 5.10. Tổng hợp bảng đặc trưng ứng viên

Cell dưới liệt kê toàn bộ feature đã tạo cùng nhóm, điều kiện khả dụng và tỷ lệ giá trị thiếu. Đây là danh sách **ứng viên an toàn để bàn giao**. Mục 6 chỉ kiểm tra chất lượng kỹ thuật; Mục 7 khóa data contract. Việc đánh giá importance và lựa chọn theo hiệu quả mô hình được thực hiện ở Notebook 06.

In [ ]:
CANDIDATE_FEATURES = list(FEATURE_CATALOG)

catalog_table = pd.DataFrame(
    [
        {
            'Feature': name,
            'Nhóm': meta['group'],
            'Khả dụng': meta['availability'],
            'Kiểu': meta['dtype'],
            'Thiếu (%)': feature_df[name].isna().mean() * 100,
            'Mô tả': meta['description'],
        }
        for name, meta in FEATURE_CATALOG.items()
    ]
)
display(catalog_table.set_index('Feature'))

print(f'Tổng số đặc trưng ứng viên: {len(CANDIDATE_FEATURES)}')
display(catalog_table['Nhóm'].value_counts().to_frame('Số đặc trưng'))
print(f'Bộ nhớ hiện tại: {feature_df.memory_usage(deep=True).sum() / 1024 ** 2:,.1f} MB')

Yêu cầu của dự án là tạo ít nhất 10 feature có ý nghĩa; bảng trên cho biết số lượng thực tế và phân bố theo tám nhóm. Cột `Thiếu (%)` khác 0 chủ yếu ở nhóm lịch sử vì các dòng đầu chuỗi chưa có đủ quá khứ. Đây là đặc điểm availability cần được Notebook 06 xử lý bằng pipeline imputation hoặc quy tắc lọc được fit chỉ trên train.

## 6. Kiểm tra chất lượng Feature trước khi bàn giao

### 6.1. Kiểm tra schema và kiểu dữ liệu

Ba mô hình ở Notebook 06 đều cần ma trận feature dạng số. Vì vậy, bước đầu kiểm tra mỗi candidate đã tồn tại, có dtype số, có tên duy nhất và không chứa giá trị vô cực. Min/max còn giúp phát hiện lỗi mã hóa, ví dụ one-hot khác `0/1` hoặc tọa độ vượt phạm vi.

Đây là kiểm tra data contract, không sử dụng target để xếp hạng feature và không fit mô hình.

In [91]:
if len(CANDIDATE_FEATURES) != len(set(CANDIDATE_FEATURES)):
    raise RuntimeError('CANDIDATE_FEATURES có tên bị trùng.')

missing_feature_columns = set(CANDIDATE_FEATURES) - set(feature_df.columns)
if missing_feature_columns:
    raise RuntimeError(f'feature_df thiếu cột: {sorted(missing_feature_columns)}')

quality_rows = []
for name in CANDIDATE_FEATURES:
    series = feature_df[name]
    is_numeric = pd.api.types.is_numeric_dtype(series)
    infinite_count = (
        int(np.isinf(series.to_numpy(dtype='float64', na_value=np.nan)).sum())
        if is_numeric else 0
    )
    quality_rows.append(
        {
            'Feature': name,
            'Nhóm': FEATURE_CATALOG[name]['group'],
            'Dtype': str(series.dtype),
            'Là kiểu số': is_numeric,
            'Thiếu': int(series.isna().sum()),
            'Thiếu (%)': float(series.isna().mean() * 100),
            'Vô cực': infinite_count,
            'Min': float(series.min()) if series.notna().any() else np.nan,
            'Max': float(series.max()) if series.notna().any() else np.nan,
        }
    )

feature_quality_table = pd.DataFrame(quality_rows).set_index('Feature')
display(feature_quality_table)

if not feature_quality_table['Là kiểu số'].all():
    bad = feature_quality_table.index[~feature_quality_table['Là kiểu số']].tolist()
    raise RuntimeError(f'Feature không phải kiểu số: {bad}')
if feature_quality_table['Vô cực'].sum() > 0:
    raise RuntimeError('Có feature chứa giá trị vô cực.')

country_ohe_columns = [name for name in CANDIDATE_FEATURES if name.startswith('country_ohe__')]
invalid_ohe = [
    name for name in country_ohe_columns
    if feature_df[name].min() < 0 or feature_df[name].max() > 1
]
if invalid_ohe:
    raise RuntimeError(f'One-hot country nằm ngoài miền 0/1: {invalid_ohe}')

print('PASS: tất cả feature tồn tại, có kiểu số, không vô cực và one-hot hợp lệ.')

,Nhóm,Dtype,Là kiểu số,Thiếu,Thiếu (%),Vô cực,Min,Max
Feature,,,,,,,,
month_sin,Thời gian,float32,True,0,0.0000,0,-1.0000,1.0000
month_cos,Thời gian,float32,True,0,0.0000,0,-1.0000,1.0000
climatic_month_sin,Thời gian,float32,True,0,0.0000,0,-1.0000,1.0000
climatic_month_cos,Thời gian,float32,True,0,0.0000,0,-1.0000,1.0000
years_since_start,Thời gian,int16,True,0,0.0000,0,0.0000,150.0000
...,...,...,...,...,...,...,...,...
country_ohe__ukraine,Phân loại quốc gia,int8,True,0,0.0000,0,0.0000,1.0000
country_ohe__united_kingdom,Phân loại quốc gia,int8,True,0,0.0000,0,0.0000,1.0000
country_ohe__united_states,Phân loại quốc gia,int8,True,0,0.0000,0,0.0000,1.0000


PASS: tất cả feature tồn tại, có kiểu số, không vô cực và one-hot hợp lệ.


Kết quả đạt yêu cầu khi tất cả dòng ở cột `Là kiểu số` bằng `True`, tổng `Vô cực` bằng 0 và xuất hiện thông báo `PASS`. Giá trị thiếu chưa bị xem là lỗi ngay tại đây vì lag/rolling có thể thiếu hợp lệ ở đầu chuỗi; phạm vi thiếu được phân tích riêng ở Mục 6.2.

### 6.2. Kiểm tra độ phủ Feature theo từng split

Độ phủ được tính riêng cho feature-fit, validation và test để phát hiện feature bị suy giảm dữ liệu theo thời gian. Bước này chỉ đếm missing, không điền giá trị và không loại dòng. Quy tắc imputation phải nằm trong pipeline của Notebook 06 và chỉ được fit trên train.

In [92]:
split_masks = {
    'train': feature_fit_mask,
    'validation': validation_period_mask,
    'test': test_period_mask,
}

coverage_rows = []
for split_name, split_mask in split_masks.items():
    split_features = feature_df.loc[split_mask, CANDIDATE_FEATURES]
    missing_rates = split_features.isna().mean().mul(100)
    for name, missing_rate in missing_rates.items():
        coverage_rows.append(
            {
                'Split': split_name,
                'Feature': name,
                'Nhóm': FEATURE_CATALOG[name]['group'],
                'Thiếu (%)': float(missing_rate),
            }
        )

feature_coverage_table = pd.DataFrame(coverage_rows)
coverage_by_group = (
    feature_coverage_table.groupby(['Split', 'Nhóm'], observed=True)['Thiếu (%)']
    .agg(['mean', 'max'])
    .rename(columns={'mean': 'Thiếu TB (%)', 'max': 'Thiếu lớn nhất (%)'})
)
display(coverage_by_group)

print('Các feature có tỷ lệ thiếu lớn nhất:')
display(
    feature_coverage_table.sort_values('Thiếu (%)', ascending=False).head(20)
)

Thiếu TB (%)  Thiếu lớn nhất (%)
Split      Nhóm                                                 
test       Bối cảnh toàn cầu          0.0000              0.0000
           Chất lượng dữ liệu         0.0000              0.0000
           Khí hậu theo vị trí        0.0000              0.0000
           Phân loại quốc gia         0.0000              0.0000
           Thời gian                  0.0000              0.0000
           Trễ / Trượt                0.0000              0.0000
           Tương tác                  0.0000              0.0000
           Địa lý                     0.0000              0.0000
train      Bối cảnh toàn cầu          0.0534              0.0534
           Chất lượng dữ liệu         0.1108              0.1108
           Khí hậu theo vị trí        0.0000              0.0000
           Phân loại quốc gia         0.0000              0.0000
           Thời gian                  0.0000              0.0000
           Trễ / Trượt                0.5905              0.9575
           Tương tác                  0.0000              0.0000
           Địa lý                     0.0000              0.0000
validation Bối cảnh toàn cầu          0.0000              0.0000
           Chất lượng dữ liệu         0.0000              0.0000
           Khí hậu theo vị trí        0.0000              0.0000
           Phân loại quốc gia         0.0000              0.0000
           Thời gian                  0.0000              0.0000
           Trễ / Trượt                0.0000              0.0000
           Tương tác                  0.0000              0.0000
           Địa lý                     0.0000              0.0000

Các feature có tỷ lệ thiếu lớn nhất:


,Split,Feature,Nhóm,Thiếu (%)
18,train,temp_lag_12,Trễ / Trượt,0.9575
21,train,temp_anomaly_lag_12,Trễ / Trượt,0.9575
19,train,temp_roll_mean_12,Trễ / Trượt,0.4634
20,train,temp_roll_std_12,Trễ / Trượt,0.4634
24,train,city_uncertainty_lag_1,Chất lượng dữ liệu,0.1108
17,train,temp_lag_1,Trễ / Trượt,0.1108
23,train,land_anomaly_lag_1,Bối cảnh toàn cầu,0.0534
22,train,land_temperature_lag_1,Bối cảnh toàn cầu,0.0534
0,train,month_sin,Thời gian,0.0000
1,train,month_cos,Thời gian,0.0000


Nhóm `static`, thời gian, địa lý và one-hot quốc gia phải có độ phủ 100%. Missing hợp lệ chủ yếu nằm ở feature `history`, do dòng đầu chuỗi hoặc tháng lịch bị khuyết không có quá khứ để tra cứu. Notebook 05 giữ nguyên các giá trị này để Notebook 06 so sánh chiến lược imputation/lọc dòng trong pipeline mà không làm rò rỉ validation hoặc test.

### 6.3. Chẩn đoán đa cộng tuyến giữa các Feature

Đa cộng tuyến được đo giữa các feature, không dùng target và không huấn luyện mô hình. Để giảm chi phí, notebook lấy mẫu từ **feature-fit** rồi liệt kê các cặp có `|Pearson r|` vượt ngưỡng cảnh báo.

Kết quả chỉ là chẩn đoán bàn giao. Notebook 05 không tự động loại một cột trong cặp; tác động thực tế phải được Notebook 06 kiểm tra bằng metric validation, hệ số Linear Regression, feature importance hoặc ablation.

In [93]:
audit_pool = feature_df.loc[feature_fit_mask, CANDIDATE_FEATURES]
if len(audit_pool) > FEATURE_AUDIT_SAMPLE_ROWS:
    audit_positions = rng.choice(
        len(audit_pool), size=FEATURE_AUDIT_SAMPLE_ROWS, replace=False
    )
    feature_audit_df = audit_pool.iloc[np.sort(audit_positions)]
else:
    feature_audit_df = audit_pool

correlation_matrix = feature_audit_df.corr()
upper_triangle = correlation_matrix.where(
    np.triu(np.ones(correlation_matrix.shape), k=1).astype(bool)
)
high_correlation_pairs = (
    upper_triangle.stack()
    .rename('Pearson r')
    .reset_index()
    .rename(columns={'level_0': 'Feature A', 'level_1': 'Feature B'})
)
high_correlation_pairs['|r|'] = high_correlation_pairs['Pearson r'].abs()
high_correlation_pairs = (
    high_correlation_pairs[
        high_correlation_pairs['|r|'] > MULTICOLLINEARITY_THRESHOLD
    ]
    .sort_values('|r|', ascending=False)
    .reset_index(drop=True)
)

print(f'Mẫu kiểm tra feature-fit: {len(feature_audit_df):,} dòng')
print(f'Ngưỡng cảnh báo: |r| > {MULTICOLLINEARITY_THRESHOLD}')
if high_correlation_pairs.empty:
    print('Không có cặp feature nào vượt ngưỡng cảnh báo.')
else:
    print(f'Có {len(high_correlation_pairs)} cặp cần Notebook 06 xem xét:')
    display(high_correlation_pairs[['Feature A', 'Feature B', 'Pearson r']])

NameError: name 'FEATURE_AUDIT_SAMPLE_ROWS' is not defined

Một cặp vượt ngưỡng không đồng nghĩa phải loại ngay một feature. Với Linear Regression, đa cộng tuyến có thể làm hệ số kém ổn định; Random Forest và XGBoost ít nhạy hơn nhưng importance có thể bị chia giữa các biến tương tự. Quyết định cuối cùng thuộc Notebook 06 sau khi so sánh hiệu năng validation.

### 6.4. Kiểm tra Leakage và tính đúng của Feature lịch sử

Mục này xác nhận ba điều kiện: không có cột cùng tháng/target-derived trong danh sách bàn giao; `temp_lag_12` thực sự trỏ tới đúng cùng vị trí ở 12 tháng trước; và không feature nào là bản sao trực tiếp của target. Đây là kiểm tra tính đúng của pipeline, không phải đánh giá mức độ quan trọng.

In [ ]:
forbidden_model_features = {
    TARGET_COLUMN,
    'major_city_average_temperature',
    'country_average_temperature',
    'land_average_temperature',
    'city_average_temperature_uncertainty',
    'temp_anomaly_vs_climatology',
    'city_name',
    'country_name',
}
forbidden_candidates = forbidden_model_features.intersection(CANDIDATE_FEATURES)
if forbidden_candidates:
    raise RuntimeError(
        f'Danh sách bàn giao chứa cột không hợp lệ: {sorted(forbidden_candidates)}'
    )

# Xác minh lag 12 theo đúng tháng lịch trên một mẫu có giá trị.
verify_pool = feature_df.loc[
    feature_df['temp_lag_12'].notna(),
    ['location_key', 'temp_lag_12'],
]
verify_size = min(5_000, len(verify_pool))
verify_sample = verify_pool.sample(n=verify_size, random_state=RANDOM_SEED)
truth_lookup = pd.Series(
    feature_df[TARGET_COLUMN].to_numpy(),
    index=feature_df['location_key'].to_numpy(),
)
expected_lag = truth_lookup.reindex(
    verify_sample['location_key'].to_numpy() - 12
).to_numpy(dtype='float64')
lag_matches = np.allclose(
    verify_sample['temp_lag_12'].to_numpy(dtype='float64'),
    expected_lag,
    equal_nan=True,
)
if not lag_matches:
    raise RuntimeError('temp_lag_12 không khớp đúng target tại t - 12 tháng.')

# Kiểm tra bản sao target trên mẫu feature-fit; không dùng tương quan để xếp hạng.
leak_sample_size = min(50_000, int(feature_fit_mask.sum()))
leak_sample = feature_df.loc[feature_fit_mask].sample(
    n=leak_sample_size,
    random_state=RANDOM_SEED,
)
direct_target_copies = []
for name in CANDIDATE_FEATURES:
    valid = leak_sample[name].notna() & leak_sample[TARGET_COLUMN].notna()
    if valid.any() and np.array_equal(
        leak_sample.loc[valid, name].to_numpy(),
        leak_sample.loc[valid, TARGET_COLUMN].to_numpy(),
    ):
        direct_target_copies.append(name)
if direct_target_copies:
    raise RuntimeError(f'Feature là bản sao trực tiếp của target: {direct_target_copies}')

print(f'PASS: không có cột cùng tháng/chuỗi gốc trong {len(CANDIDATE_FEATURES)} feature.')
print(f'PASS: temp_lag_12 khớp đúng trên {verify_size:,} dòng kiểm tra.')
print(f'PASS: không có feature nào là bản sao trực tiếp của target trên {leak_sample_size:,} dòng.')

Ba thông báo `PASS` xác nhận feature bàn giao đúng data contract và không chứa bản sao trực tiếp của target. Kiểm tra này không thể thay thế kịch bản suy luận; Notebook 06/07 vẫn phải tái tạo lag, one-hot và fit statistics bằng đúng mapping đã lưu trong metadata.

### 6.5. Những công việc được chuyển sang Notebook 06

Notebook 05 dừng ở mức **feature hợp lệ và có thể tái tạo**. Notebook 06 sẽ tiếp tục:

1. Đọc `feature_names` và split từ metadata.
2. Fit imputer/scaler trong pipeline chỉ bằng train.
3. Huấn luyện Linear Regression, Random Forest và XGBoost.
4. So sánh MAE, RMSE, R² trên validation.
5. Tính feature importance bằng phương pháp phù hợp; với Linear Regression có thể đọc hệ số sau chuẩn hóa, với mô hình cây có thể dùng built-in/permutation importance, và SHAP là phần giải thích nâng cao.
6. Thử loại feature bằng ablation hoặc quy tắc đã nêu, rồi khóa pipeline trước khi đánh giá test.

Việc tách này tránh sử dụng một mô hình tạm trong Notebook 05 để quyết định trước kết quả của Notebook 06.

## 7. Chốt tập Feature bàn giao cho Notebook 06

### 7.1. Nguyên tắc bàn giao

Notebook 05 chỉ loại các cột vi phạm data contract: chuỗi chưa mã hóa, dữ liệu cùng tháng không có tại thời điểm dự báo, target-derived feature hoặc feature xây dựng sai trật tự thời gian. Mọi candidate đã vượt kiểm tra Mục 6 được bàn giao cho Notebook 06.

Biến `HANDOFF_FEATURES` vì thế không có nghĩa là “feature đã được mô hình chứng minh là quan trọng”; nó là danh sách feature **an toàn để đưa vào thí nghiệm modeling**.

In [ ]:
HANDOFF_FEATURES = list(CANDIDATE_FEATURES)

if not HANDOFF_FEATURES:
    raise RuntimeError('Không có feature để bàn giao cho Notebook 06.')
if len(HANDOFF_FEATURES) != len(set(HANDOFF_FEATURES)):
    raise RuntimeError('HANDOFF_FEATURES có tên cột bị trùng.')

print(f'Số feature ứng viên an toàn: {len(HANDOFF_FEATURES)}')
print('Feature importance và lựa chọn theo hiệu năng: chuyển sang Notebook 06.')

Số lượng in ra phải bằng tổng candidate ở Mục 5.10. Notebook 05 không in số feature “được giữ/bị loại theo importance” vì chưa có mô hình nào được train tại đây.

### 7.2. Bảng data contract của Feature bàn giao

Bảng dưới mô tả từng feature theo nhóm, dtype, availability, tỷ lệ missing và ý nghĩa. Đây là tài liệu đầu vào cho Notebook 06 khi xây dựng preprocessing pipeline và giao diện suy luận.

In [ ]:
handoff_table = pd.DataFrame(
    [
        {
            'Feature': name,
            'Nhóm': FEATURE_CATALOG[name]['group'],
            'Dtype': str(feature_df[name].dtype),
            'Khả dụng': FEATURE_CATALOG[name]['availability'],
            'Thiếu (%)': float(feature_df[name].isna().mean() * 100),
            'Mô tả': FEATURE_CATALOG[name]['description'],
        }
        for name in HANDOFF_FEATURES
    ]
).set_index('Feature')

display(handoff_table)
display(
    handoff_table.groupby(['Nhóm', 'Khả dụng'], observed=True)
    .size()
    .to_frame('Số feature')
)

Notebook 06 phải đọc đúng danh sách và thứ tự từ metadata, không tự chọn tất cả cột số trong bảng. Cột `Khả dụng` cho biết feature nào cần lịch sử, feature nào là thuộc tính tĩnh và feature nào phụ thuộc thống kê đã fit.

### 7.3. Các cột bị loại vì lý do ngữ nghĩa

Các cột dưới đây bị loại trước khi bàn giao vì không phù hợp với kịch bản dự báo hoặc chưa được mã hóa đúng. Đây là quyết định của Feature Engineering, độc lập với feature importance và metric mô hình.

| Cột bị loại khỏi ma trận mô hình | Lý do loại |
|---|---|
| `city_name` (chuỗi gốc) | Đây là biến định danh có cardinality rất cao. Cột vẫn được giữ để tra cứu và hiển thị nhưng không đưa trực tiếp vào `X`; đặc điểm riêng của thành phố được biểu diễn bằng tọa độ, lag và climatology. |
| `country_name` (chuỗi gốc) | Ba mô hình không nhận trực tiếp chuỗi. Cột gốc được giữ để tra cứu và phân tích, còn đầu vào mô hình sử dụng các cột `country_ohe__*`. |
| `major_city_average_temperature` | Phép đo cùng tháng gần như trùng với target tại các thành phố lớn, gây rò rỉ trực tiếp. |
| `country_average_temperature` | Nhiệt độ quốc gia cùng tháng đã chứa thông tin của thành phố cần dự báo và chưa chắc tồn tại tại thời điểm suy luận. |
| `land_average_temperature` cùng tháng | Không chắc có sẵn khi dự báo tương lai; được thay bằng `land_temperature_lag_1`. |
| `city_average_temperature_uncertainty` cùng tháng | Chỉ xuất hiện cùng phép đo target; được thay bằng `city_uncertainty_lag_1`. |
| `land_max_temperature`, `land_min_temperature`, `land_and_ocean_average_temperature` | Đều là bối cảnh của cùng tháng mục tiêu nên không phù hợp với kịch bản dự báo trước một tháng. |
| `month`, `quarter`, `decade` dạng số nguyên thô | Mùa vụ đã được biểu diễn bằng cặp sin/cos và xu hướng dài hạn bằng `years_since_start`; dùng dạng thô làm sai quan hệ tuần hoàn hoặc lặp thông tin. |
| `city_temperature_iqr_outlier` | Đây là cờ phục vụ phân tích chất lượng dữ liệu, không phải thông tin chắc chắn có trước khi dự báo target. |
| `latitude_zone`, `climatic_season` | Chỉ dùng cho phân tích; thông tin đã được biểu diễn bằng latitude và mã hóa chu kỳ có độ phân giải tốt hơn. |
| `temp_anomaly_vs_climatology` | Được suy ra trực tiếp từ target của chính dòng nên chỉ dùng để tạo insight ở Mục 8, tuyệt đối không dùng làm feature. |

Bảng Markdown ghi rõ các cột không được phép xuất hiện trong `HANDOFF_FEATURES`. Những cột này bị loại do leakage, không khả dụng khi suy luận, trùng biểu diễn hoặc là chuỗi gốc; chúng không cần chờ Notebook 06 đánh giá importance.

Ngược lại, các feature hợp lệ được giữ nguyên để Notebook 06 đánh giá công bằng trên cùng train/validation/test split.

## 8. Insight mới từ tập đặc trưng

Tập đặc trưng vừa xây dựng cho phép trả lời những câu hỏi mà Notebook 04 chưa trả lời được. Mục này khai thác `temp_anomaly_vs_climatology` — phần nhiệt độ lệch khỏi mức bình thường của chính thành phố đó trong tháng đó.

Cột này **không được dùng làm đặc trưng** vì nó suy ra trực tiếp từ target. Nhưng chính vì thế nó là công cụ phân tích tốt: nó đã loại bỏ hoàn toàn ảnh hưởng của mùa vụ và của vị trí, nên phần còn lại chủ yếu là tín hiệu biến đổi khí hậu.

In [ ]:
feature_df['temp_anomaly_vs_climatology'] = (
    feature_df[TARGET_COLUMN] - feature_df['loc_month_climatology']
).astype('float32')

anomaly_by_zone_year = (
    feature_df.groupby(['latitude_zone', 'year'], observed=True)[
        'temp_anomaly_vs_climatology'
    ]
    .mean()
    .reset_index()
)

fig, ax = plt.subplots(figsize=(13, 6))
for zone, zone_data in anomaly_by_zone_year.groupby('latitude_zone', observed=True):
    smoothed = zone_data.set_index('year')['temp_anomaly_vs_climatology'].rolling(
        10, min_periods=3
    ).mean()
    ax.plot(smoothed.index, smoothed.to_numpy(), linewidth=2, label=str(zone))

ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.axvline(TRAIN_CUTOFF_YEAR, color='#8d99ae', linewidth=1.2, linestyle=':',
           label=f'Mốc train {TRAIN_CUTOFF_YEAR}')
ax.set_xlabel('Năm', fontsize=11)
ax.set_ylabel('Lệch chuẩn so với khí hậu bình thường (°C)', fontsize=11)
ax.set_title('Nhiệt độ lệch chuẩn theo vành đai vĩ độ (trung bình trượt 10 năm)\n'
             'Đã loại ảnh hưởng mùa vụ và vị trí',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(REPORT_IMAGES_DIR / '10_anomaly_by_latitude_zone.png', dpi=200,
            bbox_inches='tight')
print('Đã lưu: reports/images/10_anomaly_by_latitude_zone.png')
plt.show()

In [ ]:
def warming_rate(group: pd.DataFrame) -> float:
    """Độ dốc °C/năm của anomaly trung bình theo năm."""
    if len(group) < 10:
        return float('nan')
    slope, _ = np.polyfit(
        group['year'].to_numpy(dtype='float64'),
        group['temp_anomaly_vs_climatology'].to_numpy(dtype='float64'),
        1,
    )
    return float(slope)


warming_by_zone = (
    anomaly_by_zone_year.groupby('latitude_zone', observed=True)[
        ['year', 'temp_anomaly_vs_climatology']
    ]
    .apply(warming_rate)
    .sort_values(ascending=False)
    .to_frame('Tốc độ ấm lên (°C/năm)')
    .rename_axis('Vành đai vĩ độ')
)
warming_by_zone['Trên 100 năm (°C)'] = warming_by_zone['Tốc độ ấm lên (°C/năm)'] * 100
display(warming_by_zone)

overall_slope = warming_rate(
    feature_df.groupby('year', observed=True)['temp_anomaly_vs_climatology']
    .mean()
    .reset_index()
)
print(f'Tốc độ anomaly toàn tập: {overall_slope:.4f}°C/năm')
print('Notebook 04 ghi nhận mức thay đổi nhiệt độ thô xấp xỉ +0,027°C/năm; '
      'năm 2013 chỉ có dữ liệu đến tháng 9.')

Mục 8 chỉ tạo insight sau khi danh sách feature đã được chốt; `temp_anomaly_vs_climatology` không phải candidate và không xuất sang Notebook 06. Kết quả theo vĩ độ bổ sung góc nhìn về xu hướng sau khi đã điều chỉnh nền khí hậu theo vị trí. Khi diễn giải vẫn cần lưu ý năm 2013 chỉ có dữ liệu đến tháng 9.

## 9. Lưu dữ liệu mới

### 9.1. Chuẩn bị bảng đầu ra

Bảng đầu ra gồm khóa nghiệp vụ, `year`/`month`, target, toàn bộ `HANDOFF_FEATURES` và nhãn `data_split`.

Không lưu cột suy ra từ target, phép đo cùng tháng không có sẵn lúc dự báo hoặc khóa kỹ thuật tạm. `data_split` có ba giá trị `train`, `validation`, `test`; Notebook 06 phải dùng nhãn này thay vì chia ngẫu nhiên. `is_train_period` được giữ để tương thích và bằng 1 cho train + validation.

In [ ]:
OUTPUT_KEY_COLUMNS = [
    'observation_date', 'city_name', 'country_name', 'latitude', 'longitude',
    'year', 'month',
]

feature_df['data_split'] = np.select(
    [feature_fit_mask, validation_period_mask, test_period_mask],
    ['train', 'validation', 'test'],
    default='unassigned',
)
feature_df['is_train_period'] = train_period_mask.astype('int8')

output_columns = (
    OUTPUT_KEY_COLUMNS
    + [TARGET_COLUMN]
    + [name for name in HANDOFF_FEATURES if name not in OUTPUT_KEY_COLUMNS]
    + ['data_split', 'is_train_period']
)
output_df = feature_df[output_columns]

forbidden_output_columns = {
    'temp_anomaly_vs_climatology',
    'major_city_average_temperature',
    'country_average_temperature',
    'land_average_temperature',
    'city_average_temperature_uncertainty',
}
leaked = forbidden_output_columns.intersection(output_df.columns)
if leaked:
    raise RuntimeError(f'Bảng đầu ra chứa cột không hợp lệ: {sorted(leaked)}')
if (feature_df['data_split'] == 'unassigned').any():
    raise RuntimeError('Có dòng chưa được gán data_split.')

print(f'Bảng đầu ra: {len(output_df):,} dòng × {output_df.shape[1]} cột')
print(f'Bộ nhớ      : {output_df.memory_usage(deep=True).sum() / 1024 ** 2:,.1f} MB')
display(output_df.head())
display(output_df['data_split'].value_counts().to_frame('Số dòng'))
print('PASS: bảng đầu ra không chứa dữ liệu cùng kỳ hoặc feature suy ra từ target.')

### 9.2. Kiểm tra dung lượng và ghi file

Bảng đầu ra có hơn 5,5 triệu dòng và thêm 49 cột one-hot country nên dung lượng sẽ tăng đáng kể so với phiên bản chưa mã hóa phân loại. Không nên ghi cứng một con số ước lượng: cell dưới tính lại kích thước theo số dòng và số cột thực tế, rồi kiểm tra dung lượng trống trước khi ghi để tránh làm đầy ổ đĩa giữa chừng.

Mặc định `OUTPUT_COMPRESSION = 'gzip'`. `pandas.read_csv()` đọc file `.csv.gz` trong suốt nên Notebook 06 không cần xử lý gì thêm. Nếu cần file `.csv` thô, đặt `OUTPUT_COMPRESSION = None` ở Mục 2.3 — với điều kiện còn đủ chỗ trống.

In [ ]:
BYTES_PER_VALUE = 9          # ước lượng độ dài trung bình một giá trị trong CSV
GZIP_RATIO = 0.22            # tỷ lệ nén thực nghiệm với dữ liệu dạng số

raw_estimate = len(output_df) * output_df.shape[1] * BYTES_PER_VALUE
estimated_bytes = raw_estimate * (GZIP_RATIO if OUTPUT_COMPRESSION == 'gzip' else 1.0)
free_bytes = shutil.disk_usage(PROCESSED_DIR).free

print(f'Ước lượng kích thước file : {estimated_bytes / 1024 ** 3:,.2f} GB '
      f'({OUTPUT_COMPRESSION or "không nén"})')
print(f'Dung lượng trống          : {free_bytes / 1024 ** 3:,.2f} GB')

if free_bytes < estimated_bytes * 1.3:
    raise RuntimeError(
        f'Không đủ dung lượng trống để ghi {OUTPUT_PATH.name} '
        f'(cần ~{estimated_bytes * 1.3 / 1024 ** 3:.2f} GB, còn {free_bytes / 1024 ** 3:.2f} GB).\n'
        'Cách xử lý: đặt OUTPUT_COMPRESSION = "gzip" ở Mục 2.3, '
        'hoặc dọn bớt file lớn trong data/processed/.'
    )
print('PASS: đủ dung lượng để ghi file.')

In [ ]:
output_df.to_csv(
    OUTPUT_PATH,
    index=False,
    encoding='utf-8',
    date_format='%Y-%m-%d',
    float_format='%.4f',
    compression=OUTPUT_COMPRESSION,
)

actual_size_mb = OUTPUT_PATH.stat().st_size / 1024 ** 2
print('Đã lưu:', OUTPUT_PATH)
print(f'Kích thước thực tế: {actual_size_mb:,.1f} MB')

# Đọc lại vài dòng đầu để xác nhận file hợp lệ và đúng thứ tự cột.
verify_df = pd.read_csv(OUTPUT_PATH, nrows=5)
if list(verify_df.columns) != output_columns:
    raise RuntimeError('Thứ tự cột trong file không khớp bảng đầu ra.')
print(f'PASS: đọc lại được {verify_df.shape[1]} cột đúng thứ tự.')
display(verify_df)

`float_format='%.4f'` giới hạn 4 chữ số thập phân: dữ liệu nguồn chỉ có 3 chữ số nên không mất thông tin, mà tránh được việc `float32` in ra hàng chục chữ số vô nghĩa và làm file phình lên.

Bước đọc lại 5 dòng đầu là phép kiểm tra rẻ nhưng hiệu quả: nó xác nhận file ghi thành công, giải nén được và giữ đúng thứ tự cột mà Notebook 06 sẽ dựa vào.

### 9.3. Lưu metadata đặc trưng

Metadata là hợp đồng dữ liệu giữa Notebook 05 và Notebook 06. Ngoài danh sách/thứ tự feature, file ghi kịch bản dự báo, ba mốc chia thời gian, số dòng từng split, điều kiện availability, bảng PostgreSQL đầu ra và cảnh báo năm 2013 chưa đủ tháng.

`feature_statistics.csv.gz` lưu climatology và thống kê theo từng vị trí–tháng để tái tạo các feature `fit_stat` khi dự báo dữ liệu mới. Các baseline fallback nhỏ hơn được ghi trực tiếp trong metadata. Notebook 06/07 không được tính lại những thống kê này từ validation hoặc test.

In [ ]:
# Lưu bảng thống kê cần thiết để tái tạo feature fit_stat khi suy luận.
location_reference = (
    feature_df[
        ['location_id', 'city_name', 'country_name', 'latitude', 'longitude',
         'latitude_zone']
    ]
    .drop_duplicates('location_id')
    .copy()
)

fixed_location_stats = location_stats.copy()
fixed_location_stats['loc_mean_temperature'] = (
    fixed_location_stats['total'] / fixed_location_stats['count']
)
fixed_location_stats['loc_temperature_std'] = np.sqrt(
    np.maximum(
        (
            fixed_location_stats['total_sq']
            - fixed_location_stats['total'] ** 2 / fixed_location_stats['count']
        )
        / (fixed_location_stats['count'] - 1),
        0,
    )
)

feature_statistics = (
    loc_month_stats.assign(
        loc_month_climatology=(
            loc_month_stats['sum'] / loc_month_stats['count']
        )
    )
    .reset_index()[['location_id', 'month', 'loc_month_climatology']]
    .merge(location_reference, on='location_id', how='left', validate='many_to_one')
    .merge(
        fixed_location_stats[
            ['loc_mean_temperature', 'loc_temperature_std']
        ].reset_index(),
        on='location_id',
        how='left',
        validate='many_to_one',
    )
)
feature_statistics.to_csv(
    FEATURE_STATS_PATH,
    index=False,
    encoding='utf-8',
    float_format='%.6f',
    compression='gzip',
)

zone_fallback = (
    zone_month_stats['sum'] / zone_month_stats['count']
).to_dict()
global_fallback = (
    global_month_stats['sum'] / global_month_stats['count']
).to_dict()

SEMANTIC_EXCLUSION_REASONS = {
    'city_name (chuỗi gốc)': (
        'Định danh có cardinality rất cao; giữ để tra cứu, không đưa trực tiếp vào X.'
    ),
    'country_name (chuỗi gốc)': (
        'Giữ để tra cứu và phân tích; đầu vào model dùng các cột country_ohe__*.'
    ),
    'major_city_average_temperature': (
        'Rò rỉ trực tiếp: phép đo cùng tháng gần như trùng target khi có giá trị.'
    ),
    'country_average_temperature': (
        'Nhiệt độ quốc gia cùng tháng chứa thông tin thành phố cần dự báo.'
    ),
    'land_average_temperature (cùng tháng)': (
        'Không có sẵn khi dự báo tương lai; thay bằng land_temperature_lag_1.'
    ),
    'city_average_temperature_uncertainty (cùng tháng)': (
        'Chỉ xuất hiện cùng target; thay bằng city_uncertainty_lag_1.'
    ),
    'land_max/min/land_and_ocean_temperature': (
        'Bối cảnh cùng tháng, không phù hợp kịch bản dự báo một bước.'
    ),
    'month / quarter / decade (dạng số nguyên thô)': (
        'Mùa vụ và xu hướng đã được biểu diễn bằng sin/cos và years_since_start.'
    ),
    'city_temperature_iqr_outlier': (
        'Cờ phân tích chất lượng, không phải thông tin có sẵn khi dự báo.'
    ),
    'latitude_zone / climatic_season': (
        'Chỉ dùng phân tích; thông tin đã có trong latitude và mã hóa chu kỳ.'
    ),
    'temp_anomaly_vs_climatology': (
        'Suy ra trực tiếp từ target, chỉ dùng cho insight ở Mục 8.'
    ),
}

feature_metadata = {
    'created_at': datetime.now(timezone.utc).isoformat(timespec='seconds'),
    'created_by': '05_feature_engineering.ipynb',
    'data_source': DATA_SOURCE,
    'forecast_scenario': 'one_month_ahead_with_observed_history',
    'fast_mode': FAST_MODE,
    'row_count': int(len(output_df)),
    'location_count': int(feature_df['location_id'].nunique()),
    'target': TARGET_COLUMN,
    'feature_fit_cutoff_year': int(FEATURE_FIT_CUTOFF_YEAR),
    'validation_start_year': int(VALIDATION_START_YEAR),
    'train_cutoff_year': int(TRAIN_CUTOFF_YEAR),
    'split_counts': {
        str(name): int(count)
        for name, count in output_df['data_split'].value_counts().items()
    },
    'observation_date_range': [
        str(feature_df['observation_date'].min().date()),
        str(feature_df['observation_date'].max().date()),
    ],
    'last_year_is_partial': True,
    'last_observation_date': '2013-09-01',
    'output_file': OUTPUT_PATH.name,
    'feature_statistics_file': FEATURE_STATS_PATH.name,
    'postgres_table': f'public.{FEATURE_TABLE}',
    'key_columns': OUTPUT_KEY_COLUMNS,
    'feature_names': HANDOFF_FEATURES,
    'country_encoding': {
        'source_column': 'country_name',
        'method': 'one_hot_drop_first',
        'fit_period': f'year <= {FEATURE_FIT_CUTOFF_YEAR}',
        'reference_country': COUNTRY_REFERENCE_COUNTRY,
        'encoded_feature_to_country': COUNTRY_OHE_MAPPING,
        'unknown_policy': 'all country_ohe columns equal 0',
    },
    'feature_details': {
        name: {
            'group': FEATURE_CATALOG[name]['group'],
            'availability': FEATURE_CATALOG[name]['availability'],
            'description': FEATURE_CATALOG[name]['description'],
        }
        for name in HANDOFF_FEATURES
    },
    'fallback_statistics': {
        'zone_month_climatology': {
            f'{str(zone)}|{int(month)}': round(float(value), 6)
            for (zone, month), value in zone_fallback.items()
        },
        'global_month_climatology': {
            str(int(month)): round(float(value), 6)
            for month, value in global_fallback.items()
        },
        'global_land_month_baseline': {
            str(int(month)): round(float(value), 6)
            for month, value in global_month_baseline.items()
        },
    },
    'semantic_exclusions': SEMANTIC_EXCLUSION_REASONS,
    'model_evaluation': {
        'status': 'deferred_to_notebook_06',
        'models': ['Linear Regression', 'Random Forest', 'XGBoost'],
        'feature_importance': 'computed_in_notebook_06',
        'feature_selection': 'computed_in_notebook_06',
    },
    'notes_for_notebook_06': [
        'Dùng data_split: train để fit, validation để chọn mô hình, test để báo cáo cuối.',
        'Không dùng test để chọn feature, điều chỉnh hyperparameter hoặc preprocessing.',
        'Train model và tính feature importance hoàn toàn trong Notebook 06.',
        'Feature availability=history cần dữ liệu các tháng trước của đúng địa điểm.',
        'Năm 2013 chỉ có dữ liệu đến tháng 9; cần nêu rõ khi báo cáo test.',
        'Dùng feature_names theo đúng thứ tự khi huấn luyện và dự báo ở Notebook 06.',
        'Không đưa country_name hoặc city_name dạng chuỗi trực tiếp vào X.',
        'Dùng country_encoding để tái tạo đúng one-hot country khi suy luận.',
    ],
}

METADATA_PATH.write_text(
    json.dumps(feature_metadata, ensure_ascii=False, indent=2), encoding='utf-8'
)
print('Đã lưu:', FEATURE_STATS_PATH)
print(f'Số dòng thống kê feature: {len(feature_statistics):,}')
print('Đã lưu:', METADATA_PATH)
print(f'Số đặc trưng ghi trong metadata: {len(feature_metadata["feature_names"])}')
print(json.dumps(
    {key: feature_metadata[key] for key in (
        'target', 'forecast_scenario', 'feature_fit_cutoff_year',
        'validation_start_year', 'train_cutoff_year', 'split_counts', 'feature_names'
    )},
    ensure_ascii=False,
    indent=2,
))

### 9.4. Lưu tập mẫu cho ứng dụng

Ứng dụng Streamlit/FastAPI trong `app/` và Notebook 07 không cần đọc toàn bộ 5,5 triệu dòng. Một tập mẫu nhỏ cho phép khởi động nhanh và demo được ngay cả khi không có PostgreSQL.

Mẫu được lấy phân tầng theo vành đai vĩ độ để giữ được tính đa dạng địa lý, thay vì lấy ngẫu nhiên thuần có thể thiên lệch về các quốc gia có nhiều dòng như India hay China.

In [ ]:
zone_labels = feature_df['latitude_zone']
sample_fraction = min(1.0, SAMPLE_ROWS / len(output_df))

# GroupBy.sample lấy mẫu theo từng vành đai vĩ độ với cùng tỷ lệ, nên cấu trúc
# địa lý của mẫu giống tập đầy đủ.
sample_df = (
    output_df.groupby(zone_labels, observed=True, sort=False)
    .sample(frac=sample_fraction, random_state=RANDOM_SEED)
    .sort_index()
)
sample_df.to_csv(SAMPLE_PATH, index=False, encoding='utf-8',
                 date_format='%Y-%m-%d', float_format='%.4f')

print('Đã lưu:', SAMPLE_PATH)
print(f'Kích thước: {SAMPLE_PATH.stat().st_size / 1024 ** 2:,.2f} MB · {len(sample_df):,} dòng')
display(
    sample_df.groupby(feature_df.loc[sample_df.index, 'latitude_zone'], observed=True)
    .size()
    .to_frame('Số dòng trong mẫu')
    .rename_axis('Vành đai vĩ độ')
)

### 9.5. Nạp bảng đặc trưng vào PostgreSQL

`public.city_temperature_features` là đầu ra chính và bắt buộc của Notebook 05. Cell sử dụng `COPY ... FROM STDIN` theo chunk trong một transaction, tạo index theo ngày và chạy `ANALYZE`. Nếu nạp lỗi, transaction rollback và notebook dừng; không có nhánh bỏ qua database.

In [ ]:
import io

from sqlalchemy import text

def postgres_type(series: pd.Series) -> str:
    """Suy ra kiểu PostgreSQL từ dtype của pandas.

    Dùng các hàm kiểm tra kiểu của pandas thay vì so khớp tên dtype, vì
    pandas có thể biểu diễn ngày tháng bằng nhiều đơn vị khác nhau
    (`datetime64[ns]`, `datetime64[us]`, ...) tùy nguồn dữ liệu.
    """
    if pd.api.types.is_datetime64_any_dtype(series):
        return 'DATE'
    if pd.api.types.is_bool_dtype(series):
        return 'BOOLEAN'
    if pd.api.types.is_float_dtype(series):
        return 'DOUBLE PRECISION'
    if pd.api.types.is_integer_dtype(series):
        return 'INTEGER'
    return 'TEXT'

# Sinh DDL từ chính bảng đầu ra để schema không bị lệch với dữ liệu.
column_definitions = [
    f'    {column} {postgres_type(output_df[column])}'
    for column in output_df.columns
]

create_table_sql = (
    f'DROP TABLE IF EXISTS public.{FEATURE_TABLE};\n'
    f'CREATE TABLE public.{FEATURE_TABLE} (\n'
    + ',\n'.join(column_definitions)
    + f',\n    PRIMARY KEY ({", ".join(BUSINESS_KEY_COLUMNS)})\n);'
)
print(create_table_sql)

COPY_CHUNK_ROWS = 250_000
raw_connection = DB_ENGINE.raw_connection()
try:
    with raw_connection.cursor() as cursor:
        cursor.execute(create_table_sql)
        copy_sql = (
            f'COPY public.{FEATURE_TABLE} ({", ".join(output_df.columns)}) '
            "FROM STDIN WITH (FORMAT csv, HEADER false, NULL '')"
        )
        for start in range(0, len(output_df), COPY_CHUNK_ROWS):
            buffer = io.StringIO()
            output_df.iloc[start:start + COPY_CHUNK_ROWS].to_csv(
                buffer, index=False, header=False,
                date_format='%Y-%m-%d', float_format='%.4f',
            )
            buffer.seek(0)
            cursor.copy_expert(copy_sql, buffer)
            print(f'  đã nạp {min(start + COPY_CHUNK_ROWS, len(output_df)):,} '
                  f'/ {len(output_df):,} dòng')
        cursor.execute(
            f'CREATE INDEX idx_{FEATURE_TABLE}_date '
            f'ON public.{FEATURE_TABLE} (observation_date);'
        )
        cursor.execute(f'ANALYZE public.{FEATURE_TABLE};')
    raw_connection.commit()
except Exception:
    raw_connection.rollback()
    raise
finally:
    raw_connection.close()

with DB_ENGINE.connect() as connection:
    loaded = pd.read_sql_query(
        text(f'SELECT COUNT(*) AS row_count FROM public.{FEATURE_TABLE}'), connection
    )
display(loaded)
if int(loaded.loc[0, 'row_count']) != len(output_df):
    raise RuntimeError('Row count trong PostgreSQL không khớp bảng đầu ra.')
print(f'PASS: đã nạp đủ {len(output_df):,} dòng vào public.{FEATURE_TABLE}.')

Khi cell chạy, `row_count` trong PostgreSQL phải khớp tuyệt đối với số dòng của bảng đầu ra. Toàn bộ thao tác nằm trong một transaction: nếu bất kỳ khối nào lỗi, `rollback()` bảo đảm database không bị bỏ lại ở trạng thái nạp dở dang.

## 10. Kết luận và bàn giao cho Notebook 06

In [ ]:
summary = pd.DataFrame(
    {
        'Giá trị': [
            DATA_SOURCE,
            f'{len(feature_df):,}',
            f"{feature_df['location_id'].nunique():,}",
            f'{len(CANDIDATE_FEATURES)}',
            f'{len(HANDOFF_FEATURES)}',
            f'{TRAIN_CUTOFF_YEAR}',
            f'{OUTPUT_PATH.name} ({OUTPUT_PATH.stat().st_size / 1024 ** 2:,.1f} MB)',
            'Notebook 06',
        ]
    },
    index=[
        'Nguồn dữ liệu', 'Số dòng', 'Số vị trí',
        'Feature đã tạo', 'Feature bàn giao', 'Mốc train + validation',
        'File đầu ra', 'Train / importance / lựa chọn feature',
    ],
)
display(summary)

Notebook 05 hoàn thành bốn trách nhiệm:

1. **Xây dựng feature:** tám nhóm feature nối trực tiếp với EDA về mùa vụ, địa lý, xu hướng, lịch sử và chất lượng dữ liệu.
2. **Bảo đảm đúng thời gian:** feature động chỉ dùng quá khứ; target statistics dùng leave-one-out ở feature-fit và cố định cho validation/test.
3. **Kiểm tra data contract:** schema số, missing, one-hot, đa cộng tuyến và leakage được kiểm tra mà không train model.
4. **Bàn giao:** toàn bộ feature ứng viên hợp lệ, mapping, thống kê fit và split được lưu cho Notebook 06.

Notebook 05 không kết luận feature nào quan trọng nhất. Kết luận đó chỉ có ý nghĩa sau khi Notebook 06 huấn luyện và đánh giá các mô hình trên validation.

**Notebook 06 phải:**

- Đọc `feature_metadata.json` và dùng `feature_names` đúng thứ tự.
- Dùng `data_split`: train để fit, validation để chọn mô hình/hyperparameter/feature, test chỉ để báo cáo cuối.
- Fit imputer, scaler và mọi preprocessing chỉ trên train.
- Huấn luyện Linear Regression, Random Forest và XGBoost bằng cùng data contract.
- Đánh giá MAE, RMSE, R² và feature importance; có thể dùng permutation importance/SHAP hoặc ablation.
- Khóa pipeline trước khi mở test và lưu đầy đủ preprocessing cùng mô hình.

## 11. Checklist hoàn thành Notebook 05

- [x] Chỉ đọc dữ liệu sạch từ `public.cleaned_city_temperature` trong PostgreSQL.
- [x] Kiểm tra data contract, grain City–month và target đầy đủ.
- [x] Chia feature-fit, validation và test theo thời gian để bàn giao cho Notebook 06.
- [x] Chỉ nạp 14/32 cột thật sự cần xây dựng feature và hạ kiểu dữ liệu.
- [x] Chuyển kết quả Notebook 04 thành tám nhóm feature phù hợp bài toán nhiệt độ.
- [x] Mã hóa mùa vụ tuần hoàn có hiệu chỉnh bán cầu.
- [x] Mã hóa one-hot cho `country_name`, không one-hot `city_name`.
- [x] Tạo target statistics leave-one-out ở fit và cố định cho validation/test.
- [x] Tạo lag/rolling theo đúng tháng lịch, không dùng `shift()` theo số dòng.
- [x] Thay feature cùng tháng bằng phiên bản lag khi cần.
- [x] Kiểm tra schema số, missing, one-hot, đa cộng tuyến và leakage bằng code.
- [x] Loại các cột không hợp lệ theo ngữ nghĩa/data contract.
- [x] Không train model, không tính feature importance và không chọn feature theo metric trong Notebook 05.
- [x] Lưu feature, metadata, mapping và tập mẫu bổ sung.
- [x] Nạp bắt buộc bảng `public.city_temperature_features` bằng `COPY` và kiểm tra row count.
- [x] Bàn giao rõ nhiệm vụ train/importance/feature selection cho Notebook 06.